## notebook 1 - Analysis

In [ ]:
# ==========================================
# n1-1-1 — Project & Pipeline Configuration
# ==========================================

import yaml
from pathlib import Path

# ------------------------------------------
# a) Project metadata
# ------------------------------------------

# Name of your dataset folder (where results & figures go)
DATASET_NAME = "260430-melanoma-monocytes"

# Name of the config file (without .yaml)
CONFIG_NAME = "melanoma"

# ------------------------------------------
# b) Paths & directories
# ------------------------------------------

BASE_DIR = Path("../").resolve()
# BASE_DIR = Path.cwd().parents[1]

CONFIG_DIR = BASE_DIR / "configs"
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
DATA_DIR = BASE_DIR / "data" / DATASET_NAME
RESULTS_DIR = BASE_DIR / "results" / DATASET_NAME
FIG_DIR = BASE_DIR / "figures" / DATASET_NAME

for p in [RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------
# c) Load configuration from YAML
# ------------------------------------------

config_path = CONFIG_DIR / f"{CONFIG_NAME}.yaml"

if not config_path.exists():
    raise FileNotFoundError(
        f"Config file not found: {config_path}\n"
        f"Make sure a YAML file exists in configs/ with this name."
    )

with open(config_path, "r") as f:
    CONFIG = yaml.safe_load(f)

# ------------------------------------------
# d) Extract configuration sections
# ------------------------------------------

META = CONFIG.get("meta", {})
QC = CONFIG.get("qc", {})
ANALYSIS = CONFIG.get("analysis", {})
ANNOTATION = CONFIG.get("annotation", {})
VIS = CONFIG.get("visualization", {})

# ------------------------------------------
# e) Global settings
# ------------------------------------------

RANDOM_SEED = CONFIG.get("random_seed", 0)
MULTI_RESOLUTIONS = CONFIG.get("multi_resolutions", [0.3, 0.5, 0.8, 1.0])
RANK_GENES_METHOD = CONFIG.get("rank_genes_method", "wilcoxon")
CLUSTER_KEY = CONFIG.get("cluster_key", "leiden")

# ------------------------------------------
# f) QC parameters
# ------------------------------------------

MIN_GENES_PER_CELL = QC.get("min_genes_per_cell")
MAX_GENES_PER_CELL = QC.get("max_genes_per_cell")
MAX_PCT_COUNTS_MT = QC.get("max_pct_mt")
MIN_CELLS_PER_GENE = QC.get("min_cells_per_gene")

# ------------------------------------------
# g) Analysis parameters
# ------------------------------------------

N_TOP_HVGS = ANALYSIS.get("n_top_hvgs")
N_PCS = ANALYSIS.get("n_pcs")
NEIGHBOR_K = ANALYSIS.get("neighbor_k")
DEFAULT_LEIDEN_RESOLUTION = ANALYSIS.get("leiden_resolution")

# ------------------------------------------
# h) Annotation & markers
# ------------------------------------------

REFERENCE_MARKERS = ANNOTATION.get("reference_markers", {})
MARKER_GENES_FOR_UMAP = VIS.get("umap_marker_genes", [])

# ------------------------------------------
# i) Summary printout
# ------------------------------------------

print("====================================")
print("   Configuration Loaded Successfully")
print("====================================")
print(f"Dataset name:       {DATASET_NAME}")
print(f"Loaded config file: {config_path.name}")
print()
print("QC:")
print(f"  min_genes_per_cell: {MIN_GENES_PER_CELL}")
print(f"  max_genes_per_cell: {MAX_GENES_PER_CELL}")
print(f"  max_pct_mt:         {MAX_PCT_COUNTS_MT}")
print()
print("Analysis:")
print(f"  HVGs:       {N_TOP_HVGS}")
print(f"  PCs:        {N_PCS}")
print(f"  neighbors:  {NEIGHBOR_K}")
print(f"  leiden res: {DEFAULT_LEIDEN_RESOLUTION}")
print()
print(f"Annotation marker groups: {len(REFERENCE_MARKERS)}")
print(f"UMAP marker genes:        {len(MARKER_GENES_FOR_UMAP)}")
print("====================================")


In [ ]:
# ==========================================
# n1-1-2 — Configuration Validation
# ==========================================

def validate_config():

    print("Running configuration validation...\n")

    # -------------------------
    # a) QC parameters
    # -------------------------

    required_qc = {
        "MIN_GENES_PER_CELL": MIN_GENES_PER_CELL,
        "MAX_GENES_PER_CELL": MAX_GENES_PER_CELL,
        "MAX_PCT_COUNTS_MT": MAX_PCT_COUNTS_MT,
        "MIN_CELLS_PER_GENE": MIN_CELLS_PER_GENE
    }

    for name, value in required_qc.items():
        if value is None:
            raise ValueError(f"Missing QC parameter: {name}")

    if MIN_GENES_PER_CELL >= MAX_GENES_PER_CELL:
        raise ValueError(
            "MIN_GENES_PER_CELL must be smaller than MAX_GENES_PER_CELL")


    if MAX_PCT_COUNTS_MT is not None:
        if not (0 <= MAX_PCT_COUNTS_MT <= 100):
          raise ValueError("MAX_PCT_COUNTS_MT must be between 0 and 100") 
        

    # -------------------------
    # b) Analysis parameters
    # -------------------------

    required_analysis = {
        "N_TOP_HVGS": N_TOP_HVGS,
        "N_PCS": N_PCS,
        "NEIGHBOR_K": NEIGHBOR_K,
        "DEFAULT_LEIDEN_RESOLUTION": DEFAULT_LEIDEN_RESOLUTION
    }

    for name, value in required_analysis.items():
        if value is None:
            raise ValueError(f"Missing analysis parameter: {name}")

    if N_TOP_HVGS <= 0:
        raise ValueError("N_TOP_HVGS must be > 0")

    if N_PCS <= 0:
        raise ValueError("N_PCS must be > 0")

    if NEIGHBOR_K <= 0:
        raise ValueError("NEIGHBOR_K must be > 0")

    # -------------------------
    # c) Marker genes
    # -------------------------

    if not isinstance(REFERENCE_MARKERS, dict):
        raise TypeError("REFERENCE_MARKERS must be a dictionary")

    for celltype, genes in REFERENCE_MARKERS.items():

        if not isinstance(genes, list):
            raise TypeError(
                f"Markers for {celltype} must be a list of genes"
            )

        if len(genes) == 0:
            raise ValueError(
                f"Marker list for {celltype} is empty"
            )

    # -------------------------
    # d) UMAP markers
    # -------------------------

    if not isinstance(MARKER_GENES_FOR_UMAP, list):
        raise TypeError("MARKER_GENES_FOR_UMAP must be a list")

    # -------------------------
    # e) Directory check
    # -------------------------

    if not DATA_DIR.exists():
        print(f"Warning: dataset directory does not exist yet: {DATA_DIR}")

    print("Configuration validation passed.\n")


validate_config()


In [ ]:
# -------------------------------------------------------------
# n1-1-3 Setup, imports, paths, and reproducibility
# -------------------------------------------------------------

import os
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets
import gseapy as gp

# ------------------------------------------------------------------
# a) Scanpy settings
# ------------------------------------------------------------------
sc.settings.verbosity = 3          # 0: errors, 1: warnings, 2: info, 3: hints
sc.settings.n_jobs = 1
sc.set_figure_params(
    dpi=300,
    frameon=False,
    facecolor="white",
    format="png"
)

# ------------------------------------------------------------------
# b) Displaying Paths and Configuration Summary
# ------------------------------------------------------------------
print("Current working directory:", Path.cwd())
print("Base directory:", BASE_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Figures directory:", FIG_DIR)

print(f"\nDataset: {DATASET_NAME}")
print(f"Main clustering key: {CLUSTER_KEY}")
print(f"Default Leiden resolution: {DEFAULT_LEIDEN_RESOLUTION}")

# ------------------------------------------------------------------
# c) Software versions (for reproducibility)
# ------------------------------------------------------------------
print("\nPackage versions:")
for pkg in ["scanpy", "anndata", "pandas", "numpy", "matplotlib", "seaborn", "ipywidgets", "gseapy"]:
    try:
        ver = importlib.metadata.version(pkg)
        print(f"  - {pkg}: {ver}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  - {pkg}: NOT INSTALLED")


np.random.seed(RANDOM_SEED)
sc.settings.seed = RANDOM_SEED  



# Save enviroment

env_path = RESULTS_DIR / "0-environment.txt"
with open(env_path, "w") as f:
    for pkg in ["scanpy", "anndata", "pandas", "numpy", "matplotlib", "seaborn", "ipywidgets", "gseapy"]:
        try:
            ver = importlib.metadata.version(pkg)
            f.write(f"{pkg}=={ver}\n")
        except:
            pass


In [ ]:
# ------------------------------------------------------------------
# n1-1-4- Pretty printing of configuration (Rich formatting) (Optional)
# ------------------------------------------------------------------

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

console = Console()

# -------------------------------------------------
# a) Project / Dataset info panel
# -------------------------------------------------
header = Text()
header.append("scRNA-seq Pipeline\n", style="bold cyan")
header.append(f"Dataset: {DATASET_NAME}\n", style="bold white")
header.append(f"Clustering key: {CLUSTER_KEY}\n", style="white")
header.append(f"Default Leiden resolution: {DEFAULT_LEIDEN_RESOLUTION}", style="white")

console.print(
    Panel(
        header,
        title="Project Summary",
        border_style="cyan",
        expand=False
    )
)

# -------------------------------------------------
# b) Paths table
# -------------------------------------------------
paths_table = Table(title="Project Paths", show_header=True, header_style="bold magenta")

paths_table.add_column("Name", style="bold")
paths_table.add_column("Path", style="green")

paths_table.add_row("Base directory", str(BASE_DIR))
paths_table.add_row("Raw data", str(RAW_DATA_DIR))
paths_table.add_row("Processed data", str(DATA_DIR))
paths_table.add_row("Results", str(RESULTS_DIR))
paths_table.add_row("Figures", str(FIG_DIR))

console.print(paths_table)

# -------------------------------------------------
# c) Analysis parameters table
# -------------------------------------------------
params_table = Table(title="Key Analysis Parameters", show_header=True, header_style="bold yellow")

params_table.add_column("Parameter", style="bold")
params_table.add_column("Value", style="white")

params_table.add_row("Clustering method", "Leiden")
params_table.add_row("Cluster key", CLUSTER_KEY)
params_table.add_row("Default resolution", str(DEFAULT_LEIDEN_RESOLUTION))
params_table.add_row("Random seed", "Defined in config cell")

console.print(params_table)


In [ ]:
# ==========================================
# n1-2-1 — Universal 10x Data Loader (FINAL)
# Supports:
#   ✔ Single-sample
#   ✔ Multi-sample via sample_info.yaml
# ==========================================

import scanpy as sc
import pandas as pd
import yaml
from pathlib import Path

print("====================================")
print("      DATA LOADING STARTED")
print("====================================")

# -------------------------------------------------
# 0) Detect sample_info.yaml
# -------------------------------------------------

sample_info_path = CONFIG_DIR / "sample_info.yaml"

USE_MULTI_SAMPLE = False
SAMPLE_INFO = {}

if sample_info_path.exists():
    print(f"[INFO] Found sample info file:\n{sample_info_path}")
    
    with open(sample_info_path, "r") as f:
        sample_yaml = yaml.safe_load(f)
    
    SAMPLE_INFO = sample_yaml.get("samples", {})
    
    if isinstance(SAMPLE_INFO, dict) and len(SAMPLE_INFO) > 0:
        USE_MULTI_SAMPLE = True

print(f"[INFO] Multi-sample mode: {USE_MULTI_SAMPLE}")

# -------------------------------------------------
# 1) Multi-sample loading
# -------------------------------------------------

if USE_MULTI_SAMPLE:

    print("\n[Step 1] Loading multi-sample dataset...")

    adatas = []

    for sample_name, condition in SAMPLE_INFO.items():

        sample_path = DATA_DIR / sample_name

        print(f"\nLoading sample: {sample_name}")
        print(f"Path: {sample_path}")

        if not sample_path.exists():
            raise FileNotFoundError(
                f"Sample folder not found:\n{sample_path}"
            )

        ad = sc.read_10x_mtx(
            sample_path,
            var_names="gene_symbols",
            cache=True,
            compressed=False
        )

        ad.var_names_make_unique()

        # ----------------------------------
        # Make cell barcodes globally unique
        # ----------------------------------
        ad.obs_names = [f"{sample_name}_{bc}" for bc in ad.obs_names]

        # ----------------------------------
        # Add metadata
        # ----------------------------------
        ad.obs["sample_id"] = sample_name
        ad.obs["condition"] = condition
        ad.obs["group"] = condition

        adatas.append(ad)

    # --------------------------------------
    # Merge all samples
    # --------------------------------------
    print("\n[Step 2] Merging samples...")

    adata = sc.concat(
        adatas,
        join="outer",
        label="batch",
        keys=list(SAMPLE_INFO.keys())
    )

# -------------------------------------------------
# 2) Single-sample loading
# -------------------------------------------------

else:

    print("\n[Step 1] Loading single-sample dataset...")

    print(f"Path: {DATA_DIR}")

    if not DATA_DIR.exists():
        raise FileNotFoundError(f"DATA_DIR not found:\n{DATA_DIR}")

    adata = sc.read_10x_mtx(
        DATA_DIR,
        var_names="gene_symbols",
        cache=True,
        compressed=False
    )

    adata.var_names_make_unique()

    # ----------------------------------
    # Add minimal metadata
    # ----------------------------------
    adata.obs["sample_id"] = DATASET_NAME
    adata.obs["condition"] = "unknown"
    adata.obs["group"] = "unknown"

# -------------------------------------------------
# 3) Final cleanup
# -------------------------------------------------

print("\n[Step 3] Final cleanup...")

adata.var_names_make_unique()
adata.obs_names_make_unique()

adata.uns["dataset"] = DATASET_NAME

adata.uns["pipeline"] = {
    "dataset": DATASET_NAME,
    "config": CONFIG_NAME,
    "date": pd.Timestamp.now().isoformat(),
    "random_seed": RANDOM_SEED,
    "mode": "multi" if USE_MULTI_SAMPLE else "single"
}

print("\nFinal AnnData:")
print(adata)

# -------------------------------------------------
# 4) Save raw checkpoint ⭐
# -------------------------------------------------

print("\n[Step 4] Saving raw checkpoint...")

raw_h5ad_path = RESULTS_DIR / f"1-{DATASET_NAME}_raw.h5ad"
adata.write(raw_h5ad_path)

print(f"Raw AnnData saved to:\n{raw_h5ad_path}")

print("\n====================================")
print("      DATA LOADING COMPLETED")
print("====================================")

In [ ]:
# # n1-2-2. Detection of organism

def detect_organism_from_adata(adata, n_check=50):
    genes = list(adata.var_names[:n_check]) + list(adata.var_names[-n_check:])

    if any(g.startswith("ENSG") for g in genes):
        return "human"
    if any(g.startswith("ENSMUSG") for g in genes):
        return "mouse"

    human_like = sum(g.isupper() for g in genes)
    mouse_like = sum(g[:1].isupper() and g[1:].islower() for g in genes)

    if human_like > mouse_like:
        return "human"
    elif mouse_like > human_like:
        return "mouse"

    return "unknown"


def normalize_organism(org):
    org = org.lower().strip()

    mapping = {
        "homo sapiens": "human",
        "hsapiens": "human",
        "human": "human",
        "mus musculus": "mouse",
        "mmusculus": "mouse",
        "mouse": "mouse"
    }

    return mapping.get(org, org)


# --- detect ---
ORGANISM_DEFAULT = detect_organism_from_adata(adata)

if ORGANISM_DEFAULT == "unknown":
    print("[WARNING] Could not confidently detect organism.")
    print("[WARNING] Falling back to 'human' (please verify).")
    ORGANISM_DEFAULT = "human"

# --- normalize ---
ORGANISM_DEFAULT = normalize_organism(ORGANISM_DEFAULT)

# --- store ---
adata.uns["organism"] = ORGANISM_DEFAULT

print(f"[INFO] Organism set to: {ORGANISM_DEFAULT}")

In [ ]:
# ==========================================
# n1-3-1  — Quality Control (QC) Pipeline
# Compatible with single & multi-sample data
# ==========================================

import matplotlib.pyplot as plt

print("====================================")
print("        QC PIPELINE STARTED")
print("====================================")

# -------------------------------------------------
# 1) Identify mitochondrial genes
# -------------------------------------------------
print("\n[Step 1] Identifying mitochondrial genes...")

adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")

# -------------------------------------------------
# 2) Compute QC metrics
# -------------------------------------------------
print("[Step 2] Computing QC metrics...")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True
)

qc_cols = ["n_genes_by_counts", "total_counts", "pct_counts_mt"]

print("\nQC metrics (first rows):")
print(adata.obs[qc_cols].head())

# -------------------------------------------------
# 3) Summary statistics
# -------------------------------------------------
print("\n[Step 3] QC summary statistics...")

qc_summary = adata.obs[qc_cols].describe()
display(qc_summary)

# -------------------------------------------------
# 4) Detect if multi-sample dataset
# -------------------------------------------------
is_multi_sample = "sample_id" in adata.obs.columns
is_multi_condition = "condition" in adata.obs.columns

print("\n[Step 4] Dataset structure:")
print(f"Multi-sample:   {is_multi_sample}")
print(f"Multi-condition:{is_multi_condition}")

# -------------------------------------------------
# 5) Visualization — Global QC
# -------------------------------------------------
print("\n[Step 5] Global QC plots...")

sc.pl.violin(
    adata,
    qc_cols,
    jitter=0.4,
    multi_panel=True,
    log=True,
    show=False
)
plt.savefig(FIG_DIR / f"1-{DATASET_NAME}_QC_global_violin.png", bbox_inches="tight")
plt.close()

# -------------------------------------------------
# 6) Visualization — Per sample
# -------------------------------------------------
if is_multi_sample:
    print("[Step 6] QC per sample...")

    sc.pl.violin(
        adata,
        qc_cols,
        groupby="sample_id",
        jitter=0.4,
        multi_panel=True,
        log=True,
        show=False
    )
    plt.savefig(FIG_DIR / f"2-{DATASET_NAME}_QC_by_sample.png", bbox_inches="tight")
    plt.close()

# -------------------------------------------------
# 7) Visualization — Per condition
# -------------------------------------------------
if is_multi_condition:
    print("[Step 7] QC per condition...")

    sc.pl.violin(
        adata,
        qc_cols,
        groupby="condition",
        jitter=0.4,
        multi_panel=True,
        log=True,
        show=False
    )
    plt.savefig(FIG_DIR / f"3-{DATASET_NAME}_QC_by_condition.png", bbox_inches="tight")
    plt.close()

# -------------------------------------------------
# 8) Scatter plots
# -------------------------------------------------
print("[Step 8] Scatter plots...")

color_key = "condition" if is_multi_condition else None

sc.pl.scatter(
    adata,
    x="total_counts",
    y="pct_counts_mt",
    color=color_key,
    show=False
)
plt.savefig(FIG_DIR / f"4-{DATASET_NAME}_QC_pct_mt.png", bbox_inches="tight")
plt.close()

sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color=color_key,
    show=False
)
plt.savefig(FIG_DIR / f"5-{DATASET_NAME}_QC_n_genes.png", bbox_inches="tight")
plt.close()

# -------------------------------------------------
# 9) Top expressed genes (important diagnostic)
# -------------------------------------------------
print("[Step 9] Checking top expressed genes...")

sc.pl.highest_expr_genes(adata, n_top=20, show=False)
plt.savefig(FIG_DIR / f"6-{DATASET_NAME}_top_genes.png", bbox_inches="tight")
plt.close()

print("\n====================================")
print("        QC PIPELINE COMPLETED")
print("====================================")

In [ ]:
# ==========================================
# n1-4 — QC Filtering (robust & reusable)
# Supports:
#   ✔ Single-sample
#   ✔ Multi-sample
# ==========================================

print("====================================")
print("        QC FILTERING STARTED")
print("====================================")

# -------------------------------------------------
# 0) Store initial size
# -------------------------------------------------

n_cells_before = adata.n_obs
n_genes_before = adata.n_vars

print(f"Initial cells: {n_cells_before}")
print(f"Initial genes: {n_genes_before}")

# -------------------------------------------------
# 1) Filter cells by min genes
# -------------------------------------------------

print("\n[Step 1] Filtering cells (min_genes)...")

sc.pp.filter_cells(adata, min_genes=MIN_GENES_PER_CELL)

print(f"Cells after min_genes filter: {adata.n_obs}")

# -------------------------------------------------
# 2) Filter cells by max genes (doublets)
# -------------------------------------------------

if MAX_GENES_PER_CELL is not None:
    
    print("\n[Step 2] Filtering cells (max_genes)...")
    
    before = adata.n_obs
    
    adata = adata[
        adata.obs["n_genes_by_counts"] < MAX_GENES_PER_CELL, :
    ].copy()
    
    print(f"Removed: {before - adata.n_obs} cells")

# -------------------------------------------------
# 3) Filter cells by mitochondrial content
# -------------------------------------------------

if MAX_PCT_COUNTS_MT is not None:
    
    print("\n[Step 3] Filtering cells (mt%)...")
    
    before = adata.n_obs
    
    adata = adata[
        adata.obs["pct_counts_mt"] < MAX_PCT_COUNTS_MT, :
    ].copy()
    
    print(f"Removed: {before - adata.n_obs} cells")

# -------------------------------------------------
# 4) Filter genes
# -------------------------------------------------

print("\n[Step 4] Filtering genes (min_cells)...")

sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)

print(f"Genes after filtering: {adata.n_vars}")

# -------------------------------------------------
# 5) Summary
# -------------------------------------------------

print("\n[Step 5] QC filtering summary:")

print(f"Cells before: {n_cells_before}")
print(f"Cells after:  {adata.n_obs}")
print(f"Removed:      {n_cells_before - adata.n_obs}")

print(f"Genes before: {n_genes_before}")
print(f"Genes after:  {adata.n_vars}")

# -------------------------------------------------
# 6) Optional: check per group (VERY IMPORTANT)
# -------------------------------------------------

if "group" in adata.obs.columns:
    
    print("\n[Step 6] Cells per group after filtering:")
    print(adata.obs["group"].value_counts())

if "sample_id" in adata.obs.columns:
    
    print("\n[Step 7] Cells per sample after filtering:")
    print(adata.obs["sample_id"].value_counts())

# -------------------------------------------------
# 7) Save checkpoint ⭐
# -------------------------------------------------

print("\n[Step 8] Saving QC-filtered data...")

qc_filtered_path = RESULTS_DIR / f"2-{DATASET_NAME}_qc_filtered.h5ad"
adata.write(qc_filtered_path)

print(f"Saved to:\n{qc_filtered_path}")

print("\n====================================")
print("        QC FILTERING COMPLETED")
print("====================================")

In [ ]:
# ==========================================
# n1-5 — Normalization + Log Transform (FINAL)
# Supports:
#   ✔ Single-sample
#   ✔ Multi-sample
# ==========================================

print("====================================")
print("   NORMALIZATION & LOG TRANSFORM")
print("====================================")

# -------------------------------------------------
# 0) Store initial state
# -------------------------------------------------

print(f"Cells: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")

# -------------------------------------------------
# 1) Save raw counts (VERY IMPORTANT) ⭐
# -------------------------------------------------

print("\n[Step 1] Storing raw counts...")

adata.layers["counts"] = adata.X.copy()

# -------------------------------------------------
# 2) Normalize total counts per cell
# -------------------------------------------------

print("\n[Step 2] Normalizing total counts (CPM-like)...")

sc.pp.normalize_total(
    adata,
    target_sum=1e4,
    inplace=True
)

# -------------------------------------------------
# 3) Log-transform
# -------------------------------------------------

print("\n[Step 3] Log1p transformation...")

sc.pp.log1p(adata)

# -------------------------------------------------
# 4) Store normalized data in .raw ⭐
# -------------------------------------------------

print("\n[Step 4] Saving normalized data to .raw...")

adata.raw = adata

# -------------------------------------------------
# 5) Optional: Check normalization effect
# -------------------------------------------------

print("\n[Step 5] QC check after normalization:")

print("Total counts (should be ~1e4):")
print(adata.obs["total_counts"].head())

# -------------------------------------------------
# 6) Add pipeline metadata
# -------------------------------------------------

adata.uns["normalization"] = {
    "method": "CPM-like",
    "target_sum": 1e4,
    "log_transform": True
}

# -------------------------------------------------
# 7) Save checkpoint ⭐
# -------------------------------------------------

print("\n[Step 6] Saving normalized dataset...")

norm_log_h5ad_path = RESULTS_DIR / f"3-{DATASET_NAME}_norm_log.h5ad"
adata.write(norm_log_h5ad_path)

print(f"Saved to:\n{norm_log_h5ad_path}")

print("\n====================================")
print("   NORMALIZATION COMPLETED")
print("====================================")

In [ ]:
# ==========================================
# n1-6 — Highly Variable Genes (HVGs)
# Supports:
#   ✔ Single-sample
#   ✔ Multi-sample (batch-aware)
# ==========================================

print("====================================")
print("      HVG SELECTION STARTED")
print("====================================")

# -------------------------------------------------
# 0) Detect batch mode
# -------------------------------------------------

USE_BATCH = "sample_id" in adata.obs.columns and adata.obs["sample_id"].nunique() > 1

print(f"[INFO] Batch-aware HVG: {USE_BATCH}")

# -------------------------------------------------
# 1) Compute HVGs
# -------------------------------------------------

print("\n[Step 1] Identifying highly variable genes...")

if USE_BATCH:
    print("[INFO] Using batch-aware HVG (sample_id)")
    
    sc.pp.highly_variable_genes(
        adata,
        flavor="seurat_v3",
        n_top_genes=N_TOP_HVGS,
        batch_key="sample_id"
    )
else:
    print("[INFO] Using standard HVG (single sample)")
    
    sc.pp.highly_variable_genes(
        adata,
        flavor="seurat_v3",
        n_top_genes=N_TOP_HVGS
    )

n_hvg = int(adata.var["highly_variable"].sum())
print(f"Number of HVGs: {n_hvg}")

# -------------------------------------------------
# 2) Save HVG table (useful for debugging) ⭐
# -------------------------------------------------

print("\n[Step 2] Saving HVG table...")

hvg_table_path = RESULTS_DIR / f"4-{DATASET_NAME}_HVG_table.csv"
adata.var.to_csv(hvg_table_path)

print(f"HVG table saved to:\n{hvg_table_path}")

# -------------------------------------------------
# 3) Plot HVGs
# -------------------------------------------------

print("\n[Step 3] Plotting HVGs...")

sc.pl.highly_variable_genes(adata, show=False)

hvg_fig_path = FIG_DIR / f"7-{DATASET_NAME}_highly_variable_genes.png"
plt.savefig(hvg_fig_path, bbox_inches="tight")
plt.close()

print(f"HVG plot saved to:\n{hvg_fig_path}")

# -------------------------------------------------
# 4) Optional: check overlap across batches ⭐
# -------------------------------------------------

if USE_BATCH and "highly_variable_nbatches" in adata.var.columns:
    
    print("\n[Step 4] HVG batch distribution:")
    print(adata.var["highly_variable_nbatches"].value_counts().head())

# -------------------------------------------------
# 5) Subset to HVGs
# -------------------------------------------------

print("\n[Step 5] Subsetting to HVGs...")

adata = adata[:, adata.var["highly_variable"]].copy()

print(adata)

# -------------------------------------------------
# 6) Save checkpoint ⭐
# -------------------------------------------------

print("\n[Step 6] Saving HVG-filtered dataset...")

hvg_h5ad_path = RESULTS_DIR / f"4-{DATASET_NAME}_hvg.h5ad"
adata.write(hvg_h5ad_path)

print(f"Saved to:\n{hvg_h5ad_path}")

print("\n====================================")
print("      HVG SELECTION COMPLETED")
print("====================================")

In [ ]:
# ==========================================
# n1-7 — Scaling + PCA (FINAL)
# Supports:
#   ✔ Single-sample
#   ✔ Multi-sample
# ==========================================

print("====================================")
print("        SCALING & PCA STARTED")
print("====================================")

# -------------------------------------------------
# 0) Detect dataset type
# -------------------------------------------------

USE_BATCH = "sample_id" in adata.obs.columns and adata.obs["sample_id"].nunique() > 1

print(f"[INFO] Multi-sample: {USE_BATCH}")

# -------------------------------------------------
# 1) Optional regression (recommended) ⭐
# -------------------------------------------------

print("\n[Step 1] Regressing out unwanted variation...")

# برای پروژه شما (melanoma vs healthy):
# این کار کمک می‌کند bias ناشی از library size و mt حذف شود

sc.pp.regress_out(
    adata,
    ["total_counts", "pct_counts_mt"]
)

# -------------------------------------------------
# 2) Scaling
# -------------------------------------------------

print("\n[Step 2] Scaling data...")

sc.pp.scale(
    adata,
    max_value=10
)

# -------------------------------------------------
# 3) PCA
# -------------------------------------------------

print("\n[Step 3] Running PCA...")

sc.tl.pca(
    adata,
    svd_solver="arpack",
    n_comps=N_PCS
)

print(f"PCA computed with {N_PCS} components")

# -------------------------------------------------
# 4) Variance ratio plot
# -------------------------------------------------

print("\n[Step 4] Plotting PCA variance ratio...")

sc.pl.pca_variance_ratio(
    adata,
    log=True,
    n_pcs=N_PCS,
    show=False
)

pca_var_fig_path = FIG_DIR / f"8-{DATASET_NAME}_pca_variance_ratio.png"
plt.savefig(pca_var_fig_path, bbox_inches="tight")
plt.close()

print(f"PCA variance ratio plot saved to:\n{pca_var_fig_path}")

# -------------------------------------------------
# 5) Optional: PCA scatter (very useful) ⭐
# -------------------------------------------------

print("\n[Step 5] PCA scatter plots...")

# رنگ بر اساس condition (خیلی مهم برای شما)
if "condition" in adata.obs.columns:
    
    sc.pl.pca(
        adata,
        color="condition",
        show=False
    )
    
    pca_cond_path = FIG_DIR / f"8-{DATASET_NAME}_pca_condition.png"
    plt.savefig(pca_cond_path, bbox_inches="tight")
    plt.close()
    
    print(f"PCA (condition) saved to:\n{pca_cond_path}")

# رنگ بر اساس batch (خیلی مهم برای تشخیص batch effect)
if USE_BATCH:
    
    sc.pl.pca(
        adata,
        color="sample_id",
        show=False
    )
    
    pca_batch_path = FIG_DIR / f"8-{DATASET_NAME}_pca_batch.png"
    plt.savefig(pca_batch_path, bbox_inches="tight")
    plt.close()
    
    print(f"PCA (batch) saved to:\n{pca_batch_path}")

# -------------------------------------------------
# 6) Save checkpoint ⭐
# -------------------------------------------------

print("\n[Step 6] Saving PCA results...")

pca_h5ad_path = RESULTS_DIR / f"5-{DATASET_NAME}_pca.h5ad"
adata.write(pca_h5ad_path)

print(f"Saved to:\n{pca_h5ad_path}")

print("\n====================================")
print("        SCALING & PCA COMPLETED")
print("====================================")

In [ ]:
# ==========================================
# n1-8 & n1-9 — Neighbors + UMAP (FINAL)
# Supports:
#   ✔ Single-sample
#   ✔ Multi-sample
# ==========================================

print("====================================")
print("     NEIGHBORS + UMAP STARTED")
print("====================================")

# -------------------------------------------------
# 0) Detect dataset type
# -------------------------------------------------

USE_BATCH = "sample_id" in adata.obs.columns and adata.obs["sample_id"].nunique() > 1

print(f"[INFO] Multi-sample: {USE_BATCH}")

# -------------------------------------------------
# 1) Build neighborhood graph
# -------------------------------------------------

print("\n[Step 1] Computing neighborhood graph...")

sc.pp.neighbors(
    adata,
    n_neighbors=NEIGHBOR_K,
    n_pcs=N_PCS
)

print(f"Neighbors computed (k={NEIGHBOR_K}, PCs={N_PCS})")

# -------------------------------------------------
# 2) Save checkpoint (neighbors)
# -------------------------------------------------

print("\n[Step 2] Saving neighbors checkpoint...")

neighbors_path = RESULTS_DIR / f"6-{DATASET_NAME}_neighbors.h5ad"
adata.write(neighbors_path)

print(f"Saved to:\n{neighbors_path}")

# -------------------------------------------------
# 3) Compute UMAP
# -------------------------------------------------

print("\n[Step 3] Computing UMAP embedding...")

sc.tl.umap(
    adata,
    random_state=RANDOM_SEED
)

# -------------------------------------------------
# 4) QC-based UMAP
# -------------------------------------------------

print("\n[Step 4] Plotting QC-based UMAP...")

sc.pl.umap(
    adata,
    color=["n_genes_by_counts", "pct_counts_mt"],
    frameon=False,
    show=False
)

umap_qc_path = FIG_DIR / f"9-{DATASET_NAME}_umap_qc.png"
plt.savefig(umap_qc_path, bbox_inches="tight")
plt.close()

print(f"UMAP (QC) saved to:\n{umap_qc_path}")

# -------------------------------------------------
# 5) Biological UMAPs ⭐ مهم‌ترین بخش
# -------------------------------------------------

print("\n[Step 5] Plotting biological UMAPs...")

# --- condition (خیلی مهم برای پروژه شما)
if "condition" in adata.obs.columns:
    
    sc.pl.umap(
        adata,
        color="condition",
        frameon=False,
        show=False
    )
    
    umap_cond_path = FIG_DIR / f"9-{DATASET_NAME}_umap_condition.png"
    plt.savefig(umap_cond_path, bbox_inches="tight")
    plt.close()
    
    print(f"UMAP (condition) saved to:\n{umap_cond_path}")

# --- batch (تشخیص batch effect)
if USE_BATCH:
    
    sc.pl.umap(
        adata,
        color="sample_id",
        frameon=False,
        show=False
    )
    
    umap_batch_path = FIG_DIR / f"9-{DATASET_NAME}_umap_batch.png"
    plt.savefig(umap_batch_path, bbox_inches="tight")
    plt.close()
    
    print(f"UMAP (batch) saved to:\n{umap_batch_path}")

# -------------------------------------------------
# 6) Marker genes UMAP (اگر در config تعریف شده باشد)
# -------------------------------------------------

if len(MARKER_GENES_FOR_UMAP) > 0:
    
    print("\n[Step 6] Plotting marker genes...")

    valid_markers = [g for g in MARKER_GENES_FOR_UMAP if g in adata.var_names]

    if len(valid_markers) > 0:
        
        sc.pl.umap(
            adata,
            color=valid_markers,
            frameon=False,
            show=False
        )
        
        umap_marker_path = FIG_DIR / f"9-{DATASET_NAME}_umap_markers.png"
        plt.savefig(umap_marker_path, bbox_inches="tight")
        plt.close()
        
        print(f"UMAP (markers) saved to:\n{umap_marker_path}")
    else:
        print("[WARNING] No marker genes found in dataset")

# -------------------------------------------------
# 7) Save final checkpoint ⭐
# -------------------------------------------------

print("\n[Step 7] Saving UMAP results...")

umap_path = RESULTS_DIR / f"7-{DATASET_NAME}_umap.h5ad"
adata.write(umap_path)

print(f"Saved to:\n{umap_path}")

print("\n====================================")
print("     NEIGHBORS + UMAP COMPLETED")
print("====================================")

In [ ]:
# ==========================================
# n1-10-1 — Leiden Clustering (Main)
# ==========================================

print("====================================")
print("       LEIDEN CLUSTERING STARTED")
print("====================================")

# -------------------------------------------------
# 0) Detect dataset type
# -------------------------------------------------

USE_BATCH = "sample_id" in adata.obs.columns and adata.obs["sample_id"].nunique() > 1

print(f"[INFO] Multi-sample: {USE_BATCH}")

# -------------------------------------------------
# 1) Leiden clustering (default resolution)
# -------------------------------------------------

print(f"\n[Step 1] Running Leiden clustering (res={DEFAULT_LEIDEN_RESOLUTION})...")

sc.tl.leiden(
    adata,
    resolution=DEFAULT_LEIDEN_RESOLUTION,
    key_added=CLUSTER_KEY,
    flavor="igraph",
    n_iterations=-1,
    directed=False
)

print(f"Clusters stored in adata.obs['{CLUSTER_KEY}']")

# -------------------------------------------------
# 2) Cluster statistics ⭐
# -------------------------------------------------

print("\n[Step 2] Cluster distribution:")

cluster_counts = adata.obs[CLUSTER_KEY].value_counts().sort_index()
print(cluster_counts)

# -------------------------------------------------
# 3) UMAP — clusters
# -------------------------------------------------

print("\n[Step 3] Plotting clusters on UMAP...")

sc.pl.umap(
    adata,
    color=CLUSTER_KEY,
    legend_loc="on data",
    frameon=False,
    show=False
)

leiden_umap_path = FIG_DIR / f"10-{DATASET_NAME}_leiden_main.png"
plt.savefig(leiden_umap_path, bbox_inches="tight")
plt.close()

print(f"Saved to:\n{leiden_umap_path}")

# -------------------------------------------------
# 4) UMAP — condition (biological meaning) ⭐
# -------------------------------------------------

if "condition" in adata.obs.columns:
    
    print("\n[Step 4] Plotting condition...")

    sc.pl.umap(
        adata,
        color="condition",
        frameon=False,
        show=False
    )

    cond_path = FIG_DIR / f"10-{DATASET_NAME}_leiden_condition.png"
    plt.savefig(cond_path, bbox_inches="tight")
    plt.close()

    print(f"Saved to:\n{cond_path}")

# -------------------------------------------------
# 5) UMAP — batch (diagnostic) ⭐
# -------------------------------------------------

if USE_BATCH:
    
    print("\n[Step 5] Plotting batch effect...")

    sc.pl.umap(
        adata,
        color="sample_id",
        frameon=False,
        show=False
    )

    batch_path = FIG_DIR / f"10-{DATASET_NAME}_leiden_batch.png"
    plt.savefig(batch_path, bbox_inches="tight")
    plt.close()

    print(f"Saved to:\n{batch_path}")

# -------------------------------------------------
# 6) Save checkpoint ⭐
# -------------------------------------------------

print("\n[Step 6] Saving clustering results...")

cluster_path = RESULTS_DIR / f"8-{DATASET_NAME}_leiden.h5ad"
adata.write(cluster_path)

print(f"Saved to:\n{cluster_path}")

print("\n====================================")
print("       LEIDEN CLUSTERING COMPLETED")
print("====================================")

In [ ]:
# ==========================================
# n1-10-2 — Leiden Multi-resolution Analysis
# ==========================================

print("====================================")
print("   MULTI-RESOLUTION CLUSTERING")
print("====================================")

# -------------------------------------------------
# 1) Run multiple resolutions
# -------------------------------------------------

print("\n[Step 1] Running multiple Leiden resolutions...")

for r in MULTI_RESOLUTIONS:
    
    key = f"leiden_{r}"
    
    sc.tl.leiden(
        adata,
        resolution=r,
        key_added=key,
        flavor="igraph",
        n_iterations=-1,
        directed=False
    )
    
    print(f"✔ Resolution {r} → stored in {key}")

# -------------------------------------------------
# 2) UMAP comparison
# -------------------------------------------------

print("\n[Step 2] Plotting resolution comparison...")

sc.pl.umap(
    adata,
    color=[f"leiden_{r}" for r in MULTI_RESOLUTIONS],
    frameon=False,
    show=False
)

multi_res_path = FIG_DIR / f"11-{DATASET_NAME}_leiden_multi_res.png"
plt.savefig(multi_res_path, bbox_inches="tight")
plt.close()

print(f"Saved to:\n{multi_res_path}")

# -------------------------------------------------
# 3) Dendrogram (cluster similarity) ⭐
# -------------------------------------------------

print("\n[Step 3] Computing dendrogram...")

sc.tl.dendrogram(
    adata,
    groupby=CLUSTER_KEY
)

dendro_path = FIG_DIR / f"12-{DATASET_NAME}_dendrogram.png"

sc.pl.dendrogram(
    adata,
    groupby=CLUSTER_KEY,
    show=False
)

plt.savefig(dendro_path, bbox_inches="tight")
plt.close()

print(f"Saved to:\n{dendro_path}")

# -------------------------------------------------
# 4) Save checkpoint ⭐
# -------------------------------------------------

print("\n[Step 4] Saving multi-resolution results...")

multi_res_h5ad = RESULTS_DIR / f"9-{DATASET_NAME}_multi_resolution.h5ad"
adata.write(multi_res_h5ad)

print(f"Saved to:\n{multi_res_h5ad}")

print("\n====================================")
print("   MULTI-RESOLUTION COMPLETED")
print("====================================")

In [ ]:
# ==========================================
# n1-11 — Marker Gene Analysis (FINAL)
# Supports:
#   ✔ Single-sample
#   ✔ Multi-sample
# ==========================================

print("====================================")
print("     MARKER GENE ANALYSIS STARTED")
print("====================================")

# -------------------------------------------------
# 0) Check clustering exists
# -------------------------------------------------

if CLUSTER_KEY not in adata.obs.columns:
    raise ValueError(f"Clustering not found: {CLUSTER_KEY}")

print(f"[INFO] Using clustering key: {CLUSTER_KEY}")

# -------------------------------------------------
# 1) Rank marker genes
# -------------------------------------------------

print("\n[Step 1] Ranking marker genes...")

sc.tl.rank_genes_groups(
    adata,
    groupby=CLUSTER_KEY,
    method=RANK_GENES_METHOD,
    pts=True
)

print(f"✔ Marker analysis completed using '{RANK_GENES_METHOD}'")

# -------------------------------------------------
# 2) Plot top markers
# -------------------------------------------------

print("\n[Step 2] Plotting top marker genes...")

sc.pl.rank_genes_groups(
    adata,
    n_genes=20,
    sharey=False,
    show=False
)

marker_fig_path = FIG_DIR / f"13-{DATASET_NAME}_top_marker_genes.png"
plt.savefig(marker_fig_path, bbox_inches="tight")
plt.close()

print(f"Saved to:\n{marker_fig_path}")

# -------------------------------------------------
# 3) Export marker table ⭐ مهم
# -------------------------------------------------

print("\n[Step 3] Exporting marker table...")

markers_df = sc.get.rank_genes_groups_df(
    adata,
    group=None
)

markers_csv_path = RESULTS_DIR / f"10-{DATASET_NAME}_marker_genes.csv"
markers_df.to_csv(markers_csv_path, index=False)

print(f"Saved to:\n{markers_csv_path}")

print("\nTop rows:")
print(markers_df.head())

# -------------------------------------------------
# 4) Save per-cluster top markers ⭐
# -------------------------------------------------

print("\n[Step 4] Saving top markers per cluster...")

top_markers = (
    markers_df
    .groupby("group")
    .head(10)
)

top_markers_path = RESULTS_DIR / f"10-{DATASET_NAME}_top10_markers_per_cluster.csv"
top_markers.to_csv(top_markers_path, index=False)

print(f"Saved to:\n{top_markers_path}")

# -------------------------------------------------
# 5) Optional: Dotplot (خیلی حرفه‌ای) ⭐
# -------------------------------------------------

print("\n[Step 5] Generating dotplot...")

try:
    sc.pl.rank_genes_groups_dotplot(
        adata,
        n_genes=5,
        show=False
    )

    dotplot_path = FIG_DIR / f"14-{DATASET_NAME}_marker_dotplot.png"
    plt.savefig(dotplot_path, bbox_inches="tight")
    plt.close()

    print(f"Saved to:\n{dotplot_path}")

except Exception as e:
    print(f"[WARNING] Dotplot failed: {e}")

# -------------------------------------------------
# 6) Save processed checkpoint ⭐ مهم
# -------------------------------------------------

print("\n[Step 6] Saving processed dataset...")

processed_path = RESULTS_DIR / f"11-{DATASET_NAME}_processed_pre_annotation.h5ad"
adata.write(processed_path)

print(f"Saved to:\n{processed_path}")

print("\n====================================")
print("     MARKER GENE ANALYSIS COMPLETED")
print("====================================")

In [ ]:
## END of notebook 1- Analysis

In [ ]:
## ---------------------------------------

In [ ]:
## notebook 2 - Annotation 

In [ ]:
# ==========================================
# n2-12 — Cell Type Annotation (FINAL)
# ==========================================

print("====================================")
print("        CELL TYPE ANNOTATION")
print("====================================")

# -------------------------------------------------
# 0) Check markers
# -------------------------------------------------

if not REFERENCE_MARKERS:
    raise ValueError("REFERENCE_MARKERS is empty!")

# -------------------------------------------------
# 1) Select expression matrix
# -------------------------------------------------

if adata.raw is not None:
    expr_adata = adata.raw.to_adata()
    print("[INFO] Using adata.raw")
else:
    expr_adata = adata
    print("[INFO] Using adata")

available_genes = set(expr_adata.var_names)

# -------------------------------------------------
# 2) Compute scores
# -------------------------------------------------

print("\n[Step 1] Scoring cell types...")

celltypes = []

for ct, genes in REFERENCE_MARKERS.items():
    
    present = [g for g in genes if g in available_genes]
    
    if len(present) < 2:
        print(f"[WARN] Skipping {ct} (not enough markers)")
        continue
    
    score_key = f"score_{ct}"
    
    sc.tl.score_genes(
        expr_adata,
        gene_list=present,
        score_name=score_key,
        use_raw=False
    )
    
    adata.obs[score_key] = expr_adata.obs[score_key]
    celltypes.append(ct)

# -------------------------------------------------
# 3) Assign cell type (robust)
# -------------------------------------------------

print("\n[Step 2] Assigning cell types...")

score_cols = [f"score_{ct}" for ct in celltypes]

score_df = adata.obs[score_cols].copy()

# normalize per cell (robust)
score_df = score_df.subtract(score_df.mean(axis=1), axis=0)
score_df = score_df.divide(score_df.std(axis=1) + 1e-6, axis=0)

best_ct = score_df.idxmax(axis=1)
best_score = score_df.max(axis=1)

# adaptive threshold ⭐
threshold = best_score.quantile(0.2)

adata.obs["cell_type"] = [
    ct.replace("score_", "") if s > threshold else "Unknown"
    for ct, s in zip(best_ct, best_score)
]

# -------------------------------------------------
# 4) Cluster-level annotation
# -------------------------------------------------

print("\n[Step 3] Cluster annotation...")

cluster_map = (
    adata.obs.groupby(CLUSTER_KEY)["cell_type"]
    .agg(lambda x: x.value_counts().idxmax())
)

adata.obs["cell_type_cluster"] = adata.obs[CLUSTER_KEY].map(cluster_map)

# -------------------------------------------------
# 5) Visualization ⭐ مهم
# -------------------------------------------------

print("\n[Step 4] Plotting...")

sc.pl.umap(
    adata,
    color=["cell_type", CLUSTER_KEY],
    frameon=False,
    show=False
)

plt.savefig(FIG_DIR / f"15-{DATASET_NAME}_celltype_umap.png",
            bbox_inches="tight")
plt.close()

# marker validation
sc.pl.dotplot(
    adata,
    var_names=REFERENCE_MARKERS,
    groupby="cell_type_cluster",
    show=False
)

plt.savefig(FIG_DIR / f"16-{DATASET_NAME}_marker_validation.png",
            bbox_inches="tight")
plt.close()

# -------------------------------------------------
# 6) Save
# -------------------------------------------------

annot_path = RESULTS_DIR / f"12-{DATASET_NAME}_annotated.h5ad"
adata.write(annot_path)

print(f"\nSaved to:\n{annot_path}")
print("\n====================================")

In [ ]:
# ============================================================
# n2-13) Supplementary Annotation Quality & Marker Analysis
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

print("\n====================================")
print("   SUPPLEMENTARY ANNOTATION ANALYSIS")
print("====================================")

# --------------------------------------------------
# Step 0 — Safety checks
# --------------------------------------------------

if "cell_type" not in adata.obs:
    print("[WARNING] cell_type not found. Run annotation first.")
else:

    # --------------------------------------------------
    # Step 1 — Detect dataset structure
    # --------------------------------------------------
    
    is_multi_sample = "sample_id" in adata.obs
    is_multi_condition = "condition" in adata.obs
    
    print(f"Multi-sample:    {is_multi_sample}")
    print(f"Multi-condition: {is_multi_condition}")

    # =========================================================
    # Step 2 — Annotation confidence
    # =========================================================

    print("\n[Step 2] Computing annotation confidence...")

    score_cols = [c for c in adata.obs.columns if c.startswith("score_")]

    if len(score_cols) >= 2:
        score_matrix = adata.obs[score_cols].to_numpy()
        sorted_scores = np.sort(score_matrix, axis=1)

        confidence = sorted_scores[:, -1] - sorted_scores[:, -2]
        adata.obs["annotation_confidence"] = confidence

        print("annotation_confidence added.")
    else:
        print("[WARNING] Not enough score columns.")
        adata.obs["annotation_confidence"] = np.nan

    # =========================================================
    # Step 3 — Cluster marker genes
    # =========================================================

    print("\n[Step 3] Cluster marker validation...")

    if CLUSTER_KEY in adata.obs:

        if "rank_genes_groups" not in adata.uns:
            sc.tl.rank_genes_groups(
                adata,
                groupby=CLUSTER_KEY,
                method=RANK_GENES_METHOD,
                n_genes=50
            )

        markers_df = sc.get.rank_genes_groups_df(adata, group=None)

        markers_path = RESULTS_DIR / f"13-{DATASET_NAME}_cluster_markers.csv"
        markers_df.to_csv(markers_path, index=False)

        print(f"Markers saved: {markers_path}")

    else:
        print("[WARNING] No clustering found.")

    # =========================================================
    # Step 4 — Cluster vs Cell-Type consistency
    # =========================================================

    print("\n[Step 4] Cluster-cell type consistency...")

    if CLUSTER_KEY in adata.obs:

        ctab = pd.crosstab(
            adata.obs[CLUSTER_KEY],
            adata.obs["cell_type"]
        )

        ctab_path = RESULTS_DIR / f"14-{DATASET_NAME}_cluster_vs_celltype.csv"
        ctab.to_csv(ctab_path)

        print(f"Crosstab saved: {ctab_path}")

        # ---- dominant cell type ----
        summary = []

        for cluster, row in ctab.iterrows():
            total = row.sum()
            if total == 0:
                continue

            top_ct = row.idxmax()
            frac = row.max() / total

            summary.append({
                "cluster": cluster,
                "dominant_cell_type": top_ct,
                "fraction": frac,
                "n_cells": int(total)
            })

        summary_df = pd.DataFrame(summary)

        summary_path = RESULTS_DIR / f"15-{DATASET_NAME}_cluster_consistency.csv"
        summary_df.to_csv(summary_path, index=False)

        print(f"Consistency summary saved: {summary_path}")

    # =========================================================
    # Step 5 — UMAP visualization
    # =========================================================

    print("\n[Step 5] Plotting UMAP confidence...")

    if "X_umap" in adata.obsm:

        sc.pl.umap(
            adata,
            color="annotation_confidence",
            cmap="viridis",
            show=False
        )

        fig_path = FIG_DIR / f"17-{DATASET_NAME}_UMAP_confidence.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"UMAP confidence saved: {fig_path}")

    # =========================================================
    # Step 6 — Optional: per condition view
    # =========================================================

    if is_multi_condition:

        print("\n[Step 6] Cell-type distribution per condition:")

        print(pd.crosstab(
            adata.obs["condition"],
            adata.obs["cell_type"]
        ))

print("\n====================================")
print("   SUPPLEMENTARY ANALYSIS DONE")
print("====================================")

In [ ]:
# =========================================================
# n2-14) Automated Analysis Report (PDF) — FINAL VERSION
# =========================================================

from pathlib import Path
from datetime import datetime
import pandas as pd

# -----------------------------
# Robust fallbacks
# -----------------------------
CONFIG = globals().get("CONFIG", {})
META = globals().get("META", {})
QC = globals().get("QC", {})
ANALYSIS = globals().get("ANALYSIS", {})
ANNOTATION = globals().get("ANNOTATION", {})
VIS = globals().get("VIS", {})

DATASET_NAME = globals().get("DATASET_NAME", META.get("project_name", "dataset"))
RESULTS_DIR = globals().get("RESULTS_DIR", Path("results") / DATASET_NAME)
FIG_DIR = globals().get("FIG_DIR", Path("figures") / DATASET_NAME)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# ReportLab imports
# -----------------------------
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image,
    Table, TableStyle
)
from reportlab.lib.styles import getSampleStyleSheet

print("Building improved PDF report...")

report_path = RESULTS_DIR / f"16-{DATASET_NAME}_report_final.pdf"

doc = SimpleDocTemplate(
    str(report_path),
    pagesize=A4,
    leftMargin=2*cm, rightMargin=2*cm,
    topMargin=2*cm, bottomMargin=2*cm
)

styles = getSampleStyleSheet()
style_title = styles["Title"]
style_heading = styles["Heading2"]
style_body = styles["BodyText"]

story = []

# =========================================================
# Title
# =========================================================
story.append(Paragraph(f"scRNA-seq Report<br/>{DATASET_NAME}", style_title))
story.append(Spacer(1, 0.7*cm))

meta_text = f"""
Project: {DATASET_NAME}<br/>
Date: {datetime.now().strftime('%Y-%m-%d')}<br/>
Cells: {adata.n_obs}<br/>
Genes: {adata.n_vars}
"""
story.append(Paragraph(meta_text, style_body))
story.append(Spacer(1, 1*cm))

# =========================================================
# QC section
# =========================================================
story.append(Paragraph("1. Quality Control", style_heading))

qc_lines = []
for k, v in QC.items():
    qc_lines.append(f"{k}: {v}")

story.append(Paragraph("<br/>".join(qc_lines), style_body))
story.append(Spacer(1, 0.5*cm))

# QC images
for name in ["QC_violin", "QC_pct_counts_mt", "QC_n_genes_by_counts"]:
    p = FIG_DIR / f"{name}.png"
    if p.exists():
        story.append(Image(str(p), width=14*cm, height=7*cm))
        story.append(Spacer(1, 0.3*cm))

# =========================================================
# UMAP & Clustering (FIXED VERSION)
# =========================================================

story.append(Paragraph("2. UMAP & Clustering", style_heading))

umap_figs = [
    FIG_DIR / f"9-{DATASET_NAME}_umap_qc.png",
    FIG_DIR / f"15-{DATASET_NAME}_celltype_umap.png",
    FIG_DIR / f"9-{DATASET_NAME}_UMAP_markers.png",
    FIG_DIR / f"17-{DATASET_NAME}_UMAP_confidence.png",
]

found_any = False

for fig_path in umap_figs:
    if fig_path.exists():
        story.append(Image(str(fig_path), width=14*cm, height=7*cm))
        story.append(Spacer(1, 0.3*cm))
        print(f"[PDF] Added figure: {fig_path.name}")
        found_any = True
    else:
        print(f"[PDF WARNING] Missing figure: {fig_path.name}")

if not found_any:
    story.append(Paragraph("No UMAP figures found.", style_body))
    
# =========================================================
# Helper: compact table
# =========================================================
def make_compact_table(data, col_widths):
    tbl = Table(data, colWidths=col_widths, repeatRows=1)
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
        ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
        ("FONTSIZE", (0,0), (-1,-1), 7),   # ⭐ مهم: فونت کوچک
        ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
    ]))
    return tbl

# =========================================================
# Marker genes table
# =========================================================
story.append(Paragraph("3. Top Marker Genes", style_heading))

markers_csv = RESULTS_DIR / f"13-{DATASET_NAME}_cluster_markers.csv"

if markers_csv.exists():
    df = pd.read_csv(markers_csv)

    df = df.groupby("group").head(3)

    table_data = [["cluster", "gene", "logFC", "p_adj"]]

    for _, r in df.iterrows():
        gene = str(r.get("names", ""))[:15]  # کوتاه‌سازی
        table_data.append([
            str(r.get("group")),
            gene,
            f"{r.get('logfoldchanges',0):.2f}",
            f"{r.get('pvals_adj',0):.1e}"
        ])

    tbl = make_compact_table(
        table_data,
        col_widths=[2*cm, 3*cm, 2*cm, 2.5*cm]
    )

    story.append(tbl)
    story.append(Spacer(1, 0.5*cm))

# =========================================================
# Cluster consistency table
# =========================================================
story.append(Paragraph("4. Cluster Annotation Consistency", style_heading))

summary_csv = RESULTS_DIR / f"15-{DATASET_NAME}_cluster_consistency.csv"

if summary_csv.exists():
    df = pd.read_csv(summary_csv)

    table_data = [["cluster", "cell type", "n_cells", "fraction"]]

    for _, r in df.iterrows():
        table_data.append([
            str(r["cluster"]),
            str(r["dominant_cell_type"])[:15],
            int(r["n_cells"]),
            f"{r['fraction']:.2f}"
        ])

    tbl = make_compact_table(
        table_data,
        col_widths=[2*cm, 3*cm, 2*cm, 2.5*cm]
    )

    story.append(tbl)
    story.append(Spacer(1, 0.5*cm))

# =========================================================
# DEG explanation section ⭐ جدید
# =========================================================
story.append(Paragraph("5. Differential Expression & GO Analysis", style_heading))

deg_text = """
Differential expression (DEG) analysis identifies genes that are significantly up- or down-regulated 
between clusters or biological conditions (e.g. healthy vs disease). 

These DEGs are subsequently used for Gene Ontology (GO) enrichment analysis to identify biological processes, 
molecular functions, and cellular components associated with each cluster or condition.

Typical workflow:
1. Identify marker genes (cluster vs rest)
2. Perform DEG (condition vs condition)
3. Run GO enrichment on significant genes

Thus, DEG provides the gene list, and GO provides biological interpretation.
"""

story.append(Paragraph(deg_text, style_body))
story.append(Spacer(1, 0.5*cm))

# =========================================================
# Parameters
# =========================================================
story.append(Paragraph("6. Parameters", style_heading))

param_lines = []
for k,v in QC.items():
    param_lines.append(f"QC - {k}: {v}")

for k,v in ANALYSIS.items():
    param_lines.append(f"Analysis - {k}: {v}")

story.append(Paragraph("<br/>".join(param_lines), style_body))

# =========================================================
# Build
# =========================================================
doc.build(story)

print(f"Final PDF saved to:\n{report_path}")

In [ ]:
## END of notebook 2- Annotation

In [ ]:
# --------------------------------------------

In [ ]:
## notebook 3 - Differential Expression (DEG)

In [ ]:
# =========================================================
# n3-15) Differential Expression (DEG)
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("\n====================================")
print("        DEG ANALYSIS STARTED")
print("====================================")

# -------------------------------------------------
# Step 0) Detect dataset structure
# -------------------------------------------------

is_multi_sample = "sample_id" in adata.obs
is_multi_condition = "condition" in adata.obs and adata.obs["condition"].nunique() > 1

print(f"\n[INFO] Multi-sample: {is_multi_sample}")
print(f"[INFO] Multi-condition: {is_multi_condition}")

# -------------------------------------------------
# Step 1) Decide DEG mode
# -------------------------------------------------

if is_multi_condition:
    DEG_MODE = "condition"
    groupby_key = "condition"
    print("\n[MODE] Performing DEG between conditions (e.g. healthy vs melanoma)")
else:
    DEG_MODE = "cluster"
    groupby_key = CLUSTER_KEY
    print("\n[MODE] Single-condition dataset → DEG between clusters")

# -------------------------------------------------
# Step 2) Run DEG
# -------------------------------------------------

sc.tl.rank_genes_groups(
    adata,
    groupby=groupby_key,
    method=RANK_GENES_METHOD,
    pts=True
)

print(f"[Step 2] DEG completed using '{groupby_key}'")

# -------------------------------------------------
# Step 3) Extract DEG table
# -------------------------------------------------

deg_df = sc.get.rank_genes_groups_df(
    adata,
    group=None
)

# مرتب‌سازی بهتر
deg_df = deg_df.sort_values(
    ["group", "pvals_adj", "logfoldchanges"],
    ascending=[True, True, False]
)

# ذخیره
deg_csv_path = RESULTS_DIR / f"17-{DATASET_NAME}_DEG_results.csv"
deg_df.to_csv(deg_csv_path, index=False)

print(f"[Step 3] DEG results saved to:\n{deg_csv_path}")

# -------------------------------------------------
# Step 4) Basic filtering (recommended genes)
# -------------------------------------------------

DEG_PVAL_CUTOFF = 0.05
DEG_LOGFC_CUTOFF = 0.25

deg_filtered = deg_df[
    (deg_df["pvals_adj"] < DEG_PVAL_CUTOFF) &
    (abs(deg_df["logfoldchanges"]) > DEG_LOGFC_CUTOFF)
]

deg_filtered_path = RESULTS_DIR / f"17-{DATASET_NAME}_DEG_filtered.csv"
deg_filtered.to_csv(deg_filtered_path, index=False)

print(f"[Step 4] Filtered DEG saved to:\n{deg_filtered_path}")
print(f"Number of significant genes: {len(deg_filtered)}")

# -------------------------------------------------
# Step 5) DEG visualization (Top genes)
# -------------------------------------------------

print("\n[Step 5] Plotting DEG heatmap / dotplot...")

try:
    sc.pl.rank_genes_groups(
        adata,
        n_genes=20,
        sharey=False,
        show=False
    )

    deg_fig_path = FIG_DIR / f"18-{DATASET_NAME}_DEG_top_genes.png"
    plt.savefig(deg_fig_path, bbox_inches="tight")
    plt.close()

    print(f"DEG plot saved to:\n{deg_fig_path}")

except Exception as e:
    print(f"[WARNING] Could not generate DEG plot: {e}")

# -------------------------------------------------
# Step 6) Save DEG checkpoint
# -------------------------------------------------

deg_h5ad_path = RESULTS_DIR / f"17-{DATASET_NAME}_adata_DEG.h5ad"
adata.write(deg_h5ad_path)

print(f"\n[Step 6] DEG AnnData saved to:\n{deg_h5ad_path}")

print("\n====================================")
print("        DEG ANALYSIS COMPLETED")
print("====================================")

In [ ]:
## END of notebook 3 - Differential Expression (DEG)

In [ ]:
## --------------------------------------------

In [ ]:
## notebook 4 - GO Enrichment Analysis (BP, MF, CC) 

In [ ]:
# =========================================================
# n4-16) GO Enrichment — Enrichr Connectivity & Setup
# =========================================================

import time
import requests
import gseapy as gp
from pathlib import Path

print("\n====================================")
print("     GO ENRICHMENT INITIALIZATION")
print("====================================")

# -------------------------------------------------
# Step 0) Basic settings
# -------------------------------------------------

GO_GENE_SETS = [
    "GO_Biological_Process_2021",
    "GO_Molecular_Function_2021",
    "GO_Cellular_Component_2021",
]

PVAL_THRESH_DEFAULT = 0.05
TOP_N_DEFAULT = 100

ORGANISM = adata.uns.get("organism", "human")

ENRICHR_CHECK_TIMEOUT = 15
ENRICHR_SINGLE_TIMEOUT = 5
ENRICHR_TEST_URL = "https://maayanlab.cloud/Enrichr/datasetStatistics"

ENRICHR_AVAILABLE = False

print(f"[INFO] Organism: {ORGANISM}")
print(f"[INFO] GO gene sets: {GO_GENE_SETS}")

# -------------------------------------------------
# Step 1) Detect dataset structure
# -------------------------------------------------

is_multi_condition = "condition" in adata.obs and adata.obs["condition"].nunique() > 1

if is_multi_condition:
    DEG_MODE = "condition"
    GROUP_KEY = "condition"
    print("[MODE] GO will be based on CONDITION (healthy vs disease)")
else:
    DEG_MODE = "cluster"
    GROUP_KEY = CLUSTER_KEY
    print("[MODE] GO will be based on CLUSTERS")

# -------------------------------------------------
# Step 2) Enrichr connectivity check
# -------------------------------------------------

def _try_enrichr_connection(timeout=ENRICHR_SINGLE_TIMEOUT):
    try:
        resp = requests.get(ENRICHR_TEST_URL, timeout=timeout)
        return resp.status_code == 200
    except Exception:
        return False

print(f"[INFO] Checking Enrichr connectivity (timeout={ENRICHR_CHECK_TIMEOUT}s)...")

start_time = time.time()

while (time.time() - start_time) < ENRICHR_CHECK_TIMEOUT:
    if _try_enrichr_connection():
        ENRICHR_AVAILABLE = True
        break
    time.sleep(0.5)

# -------------------------------------------------
# Step 3) Final status
# -------------------------------------------------

if ENRICHR_AVAILABLE:
    print("[INFO] ✅ Enrichr is reachable.")
else:
    print("[WARNING] ❌ Enrichr not reachable.")
    print("[WARNING] GO enrichment will be skipped.")

# -------------------------------------------------
# Step 4) Load DEG (critical integration)
# -------------------------------------------------

deg_filtered_path = RESULTS_DIR / f"17-{DATASET_NAME}_DEG_filtered.csv"

if not deg_filtered_path.exists():
    print("[ERROR] DEG filtered file not found!")
    print("Run DEG step (n3-17) first.")
else:
    import pandas as pd
    deg_filtered = pd.read_csv(deg_filtered_path)
    print(f"[INFO] Loaded DEG filtered: {deg_filtered.shape[0]} genes")

# -------------------------------------------------
# Step 5) Prepare gene lists per group
# -------------------------------------------------

gene_lists = {}

if 'deg_filtered' in locals():

    for group in deg_filtered["group"].unique():
        sub = deg_filtered[deg_filtered["group"] == group]

        genes = sub.sort_values("logfoldchanges", ascending=False)["names"].head(TOP_N_DEFAULT).tolist()

        gene_lists[group] = genes

    print(f"[INFO] Prepared gene lists for {len(gene_lists)} groups")

# -------------------------------------------------
# Step 6) Save gene lists (for debugging / reproducibility)
# -------------------------------------------------

go_input_dir = RESULTS_DIR / f"18-{DATASET_NAME}_GO_inputs"
go_input_dir.mkdir(parents=True, exist_ok=True)

for group, genes in gene_lists.items():
    out_path = go_input_dir / f"{group}_genes.txt"
    with open(out_path, "w") as f:
        f.write("\n".join(genes))

print(f"[INFO] GO input gene lists saved to:\n{go_input_dir}")

print("\n====================================")
print("     GO SETUP COMPLETED")
print("====================================")

In [ ]:
# =============================================================================
# n4-17) GO Enrichment Function (Robust + Universal)
# =============================================================================

def run_go_enrichment_for_group(
    gene_table,
    group_name,
    gene_col="names",
    pval_col="pvals_adj",
    logfc_col="logfoldchanges",
    pval_thresh=0.05,
    top_n=100,
    organism=None,
    gene_sets=None,
    verbose=True
):

    if organism is None:
        organism = adata.uns.get("organism", "human")

    if gene_sets is None:
        gene_sets = [
            "GO_Biological_Process_2021",
            "GO_Molecular_Function_2021",
            "GO_Cellular_Component_2021",
        ]

    if verbose:
        print("\n" + "="*60)
        print(f"[GO - {group_name}] START")
        print("="*60)

    # --------------------------------------------------
    # 1) Validate input
    # --------------------------------------------------
    if gene_table is None or gene_table.empty:
        print(f"[GO - {group_name}] ❌ Empty input")
        return None

    required_cols = [gene_col, pval_col]
    for col in required_cols:
        if col not in gene_table.columns:
            raise ValueError(f"{col} not found in gene_table")

    # --------------------------------------------------
    # 2) Filtering
    # --------------------------------------------------
    df = gene_table.sort_values(pval_col)

    if logfc_col in df.columns:
        df = df[(df[pval_col] <= pval_thresh) & (df[logfc_col] > 0)]
    else:
        df = df[df[pval_col] <= pval_thresh]

    if df.empty:
        print(f"[GO - {group_name}] ❌ No genes after filtering")
        return None

    df = df.head(top_n)

    genes = df[gene_col].dropna().astype(str).unique().tolist()

    if len(genes) < 10:
        print(f"[GO - {group_name}] ❌ Too few genes: {len(genes)}")
        return None

    # --------------------------------------------------
    # 3) Enrichr availability
    # --------------------------------------------------
    if not globals().get("ENRICHR_AVAILABLE", False):
        print(f"[GO - {group_name}] ⚠️ Enrichr not available")
        return None

    # --------------------------------------------------
    # 4) Run enrichment
    # --------------------------------------------------
    import gseapy as gp

    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=gene_sets,
            organism=organism,
            outdir=None,
            cutoff=0.05
        )
    except Exception as e:
        print(f"[GO - {group_name}] ❌ Error: {e}")
        return None

    if enr is None or enr.results is None or enr.results.empty:
        print(f"[GO - {group_name}] ⚠️ No enrichment found")
        return None

    # --------------------------------------------------
    # 5) Format results
    # --------------------------------------------------
    res = enr.results.copy()

    res["group"] = group_name
    res["n_input_genes"] = len(genes)
    res["organism"] = organism

    if "Gene_set" in res.columns:
        res["go_library"] = res["Gene_set"]

    print(f"[GO - {group_name}] ✅ {res.shape[0]} terms")

    return res


print("[INFO] GO function ready.")

In [ ]:
# =============================================================================
# n4-18) Run GO Enrichment (Universal Pipeline)
# =============================================================================

print("\n[STEP 18] GO enrichment started...")

# --------------------------------------------------
# 1) Detect DE results source
# --------------------------------------------------
if "markers_df" not in globals():
    print("[ERROR] markers_df not found.")
    groups_to_analyze = []

else:
    print(f"[INFO] markers_df shape: {markers_df.shape}")

    if markers_df.empty:
        print("[ERROR] markers_df is empty")
        groups_to_analyze = []

    else:
        # حالت cluster-based
        if {"group", "names", "pvals_adj"}.issubset(markers_df.columns):
            group_col = "group"
            gene_col = "names"
            pval_col = "pvals_adj"
            print("[INFO] Using cluster-based DEGs")

        # حالت cell-type-based
        elif {"celltype", "names", "pvals_adj"}.issubset(markers_df.columns):
            group_col = "celltype"
            gene_col = "names"
            pval_col = "pvals_adj"
            print("[INFO] Using cell-type DEGs")

        else:
            raise ValueError(f"Unknown DEG format: {markers_df.columns}")

        groups_to_analyze = sorted(markers_df[group_col].dropna().unique())

print(f"[INFO] Groups: {groups_to_analyze}")

# --------------------------------------------------
# 2) Run GO per group
# --------------------------------------------------
all_results = []

for g in groups_to_analyze:

    print("\n" + "-"*60)
    print(f"[RUN] {g}")
    print("-"*60)

    sub = markers_df[markers_df[group_col] == g]

    res = run_go_enrichment_for_group(
        gene_table=sub,
        group_name=str(g),
        gene_col=gene_col,
        pval_col=pval_col,
        organism=adata.uns.get("organism", "human"),
    )

    if res is not None:
        all_results.append(res)

        out_file = RESULTS_DIR / f"18-{DATASET_NAME}_GO_{g}.csv"
        res.to_csv(out_file, index=False)
        print(f"[SAVE] {out_file}")

# --------------------------------------------------
# 3) Combine results
# --------------------------------------------------
if all_results:

    go_all = pd.concat(all_results, ignore_index=True)

    # مرتب‌سازی
    if "Adjusted P-value" in go_all.columns:
        go_all = go_all.sort_values(["group", "Adjusted P-value"])

    # FDR global
    try:
        from statsmodels.stats.multitest import multipletests

        go_all["p_adj_global"] = multipletests(
            go_all["Adjusted P-value"],
            method="fdr_bh"
        )[1]

    except Exception:
        print("[WARNING] statsmodels not available")

    out_all = RESULTS_DIR / f"19-{DATASET_NAME}_GO_all.csv"
    go_all.to_csv(out_all, index=False)

    print("\n[FINAL OUTPUT]")
    print(out_all)

    print("\n[SUMMARY]")
    print(go_all.groupby("group").size())

else:
    print("[WARNING] No GO results generated")

In [ ]:
# =============================================================================
# n4-19) GO Visualization (Universal + Robust)
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

print("\n[STEP 19] GO visualization started...")

# --------------------------------------------------
# 0) Load GO table safely
# --------------------------------------------------
if "go_all_df" in globals():
    df_go = go_all_df.copy()
    print("[INFO] Using go_all_df from memory")

else:
    go_file = RESULTS_DIR / f"19-{DATASET_NAME}_GO_all.csv"

    if go_file.exists():
        df_go = pd.read_csv(go_file)
        print(f"[INFO] Loaded GO results from file: {go_file}")
    else:
        print("[ERROR] GO results not found → run enrichment first")
        df_go = None

# --------------------------------------------------
# 1) Validation
# --------------------------------------------------
if df_go is None or df_go.empty:
    print("[STOP] No GO data available.")
    
else:

    required_cols = ["Term", "Adjusted P-value", "group"]

    missing = [c for c in required_cols if c not in df_go.columns]
    if missing:
        raise ValueError(f"Missing columns in GO table: {missing}")

    print(f"[INFO] GO table shape: {df_go.shape}")

    # --------------------------------------------------
    # 2) Preprocessing
    # --------------------------------------------------
    df_go["log10_padj"] = -np.log10(df_go["Adjusted P-value"] + 1e-300)

    def extract_go_category(x):
        x = str(x)
        if "Biological_Process" in x:
            return "BP"
        elif "Molecular_Function" in x:
            return "MF"
        elif "Cellular_Component" in x:
            return "CC"
        return "Other"

    df_go["GO_cat"] = df_go.get("go_library", "").apply(extract_go_category)

    def extract_gene_count(x):
        try:
            return int(str(x).split("/")[0])
        except:
            return 1

    if "Overlap" in df_go.columns:
        df_go["gene_count"] = df_go["Overlap"].apply(extract_gene_count)
    else:
        df_go["gene_count"] = 5  # fallback

    def shorten(term, max_len=60):
        term = str(term)
        return term if len(term) <= max_len else term[:max_len-3] + "..."

    # --------------------------------------------------
    # 3) Plot config
    # --------------------------------------------------
    color_map = {
        "BP": "tab:blue",
        "MF": "tab:orange",
        "CC": "tab:green",
        "Other": "gray"
    }

    legend_elements = [
        Line2D([0], [0], marker='o', color='w',
               label=cat,
               markerfacecolor=color_map[cat],
               markersize=8)
        for cat in ["BP", "MF", "CC"]
    ]

    groups = sorted(df_go["group"].dropna().unique())
    print(f"[INFO] Groups detected: {groups}")

    fig_counter = 19   # شروع طبق درخواست شما

    # --------------------------------------------------
    # 4) Per-group bubble plots
    # --------------------------------------------------
    for g in groups:

        df_g = df_go[df_go["group"] == g].copy()

        if df_g.empty:
            continue

        df_g = df_g.sort_values("Adjusted P-value").head(10)
        df_g = df_g.iloc[::-1]
        df_g["Term_short"] = df_g["Term"].apply(shorten)

        plt.figure(figsize=(7, 4))

        plt.scatter(
            df_g["log10_padj"],
            range(len(df_g)),
            s=df_g["gene_count"] * 25,
            c=df_g["GO_cat"].map(color_map),
            alpha=0.85
        )

        plt.yticks(range(len(df_g)), df_g["Term_short"])
        plt.xlabel("-log10(adj p-value)")
        plt.title(f"GO Bubble Plot — {g}")

        plt.legend(handles=legend_elements, title="GO Category")

        out_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_group_{g}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[FIG] {out_path}")
        fig_counter += 1

    # --------------------------------------------------
    # 5) Combined plot (important for multi-sample)
    # --------------------------------------------------
    if len(groups) > 1:

        df_top = (
            df_go
            .sort_values("Adjusted P-value")
            .groupby("group")
            .head(5)
        )

        df_top["Term_short"] = df_top["Term"].apply(shorten)

        plt.figure(figsize=(9, 5))

        plt.scatter(
            df_top["group"],
            df_top["Term_short"],
            s=df_top["gene_count"] * 25,
            c=df_top["GO_cat"].map(color_map),
            alpha=0.8
        )

        plt.xlabel("Group")
        plt.ylabel("GO Term")
        plt.title("GO Enrichment Across Groups")

        plt.legend(handles=legend_elements, title="GO Category")

        out_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_combined.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[FIG] {out_path}")
        fig_counter += 1

    # --------------------------------------------------
    # 6) Category barplots
    # --------------------------------------------------
    for cat in ["BP", "MF", "CC"]:

        df_cat = df_go[df_go["GO_cat"] == cat]

        if df_cat.empty:
            continue

        df_cat = df_cat.sort_values("Adjusted P-value").head(15)
        df_cat["Term_short"] = df_cat["Term"].apply(shorten)

        plt.figure(figsize=(7, 5))

        plt.barh(
            df_cat["Term_short"],
            df_cat["log10_padj"]
        )

        plt.xlabel("-log10(adj p-value)")
        plt.title(f"Top {cat} Terms")

        out_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_{cat}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[FIG] {out_path}")
        fig_counter += 1

    print("\n[STEP 19 COMPLETED] GO visualization done.")

In [ ]:
##  End of notebook 4 - GO Enrichment Analysis (BP, MF, CC) 

In [ ]:
## ---------------------------------------------

In [ ]:
## Notebook 5 - pathway analysis (KEGG / Reactome)

In [ ]:
# n5-20) Pathway Analysis Setup (ROBUST & FINAL)

from pathlib import Path
import time
import requests
import pandas as pd

print("\n[STEP 19] Initializing Pathway Analysis...")

# -----------------------------
# Globals
# -----------------------------
DATASET_NAME = globals().get("DATASET_NAME", "dataset")
RESULTS_DIR = globals().get("RESULTS_DIR", Path("results") / DATASET_NAME)
FIG_DIR = globals().get("FIG_DIR", Path("figures") / DATASET_NAME)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Parameters
# -----------------------------
PATHWAY_GENE_SETS = ["KEGG_2021_Human", "Reactome_2022"]
PVAL_THRESH = 0.05
TOP_N = 100
ORGANISM = globals().get("adata", {}).uns.get("organism", "human") if "adata" in globals() else "human"

# -----------------------------
# Connectivity check (ROBUST)
# -----------------------------
ENRICHR_URL = "https://maayanlab.cloud/Enrichr/datasetStatistics"

def check_enrichr():
    try:
        r = requests.get(ENRICHR_URL, timeout=5)
        return r.status_code == 200
    except Exception:
        return False

print("[INFO] Checking Enrichr connection...")

ENRICHR_AVAILABLE = False
for i in range(3):
    if check_enrichr():
        ENRICHR_AVAILABLE = True
        break
    time.sleep(1)

# 🔴 SINGLE SOURCE OF TRUTH
PATHWAY_ENRICHR_AVAILABLE = ENRICHR_AVAILABLE

if PATHWAY_ENRICHR_AVAILABLE:
    print("[INFO] ✅ Enrichr reachable")
else:
    print("[WARNING] ❌ Enrichr NOT reachable")

# -----------------------------
# DEG input
# -----------------------------
markers_df = globals().get("markers_df", None)

if markers_df is None or markers_df.empty:
    print("[ERROR] markers_df missing → run DEG step")
    PATHWAY_READY = False
else:
    PATHWAY_READY = True
    print(f"[INFO] markers_df shape: {markers_df.shape}")

# -----------------------------
# Detect structure
# -----------------------------
if PATHWAY_READY:

    if {"group", "names", "pvals_adj"}.issubset(markers_df.columns):
        GROUP_COL = "group"
    elif {"celltype", "names", "pvals_adj"}.issubset(markers_df.columns):
        GROUP_COL = "celltype"
    else:
        print("[ERROR] Unknown DEG format")
        PATHWAY_READY = False

GENE_COL = "names"
PVAL_COL = "pvals_adj"

# -----------------------------
# Groups
# -----------------------------
GROUPS = sorted(markers_df[GROUP_COL].dropna().unique()) if PATHWAY_READY else []

print(f"[INFO] Groups detected: {len(GROUPS)}")

print("\n[STEP 19 COMPLETED]\n")



In [ ]:
# n5-21) Helper function (FINAL)

def run_pathway_enrichment_for_group(gene_table, group_name):

    import gseapy as gp

    if not PATHWAY_ENRICHR_AVAILABLE:
        print(f"[SKIP] {group_name} → No internet")
        return None

    df = gene_table.sort_values(PVAL_COL)

    if "logfoldchanges" in df.columns:
        df = df.query(f"{PVAL_COL} <= @PVAL_THRESH and logfoldchanges > 0")
    else:
        df = df.query(f"{PVAL_COL} <= @PVAL_THRESH")

    df = df.head(TOP_N)

    genes = df[GENE_COL].dropna().unique().tolist()

    if len(genes) < 10:
        print(f"[SKIP] {group_name} → too few genes")
        return None

    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=PATHWAY_GENE_SETS,
            organism=ORGANISM,
            outdir=None
        )
    except Exception as e:
        print(f"[ERROR] {group_name}: {e}")
        return None

    if enr.results is None or enr.results.empty:
        return None

    res = enr.results.copy()

    if "Adjusted P-value" not in res.columns:
        for c in ["Adjusted P-value", "P-value"]:
            if c in res.columns:
                res["Adjusted P-value"] = res[c]

    res["group"] = group_name

    return res

In [ ]:
# n5-22) Run Pathway enrichment (FINAL)

print("\n[STEP 20] Running pathway enrichment...\n")

all_pathway_results = []

if not PATHWAY_READY:
    print("[ERROR] Pathway not ready")

elif not PATHWAY_ENRICHR_AVAILABLE:
    print("[WARNING] ❌ No internet → Skipping enrichment")

else:

    for g in GROUPS:

        print(f"[RUN] {g}")

        sub = markers_df[markers_df[GROUP_COL] == g]

        res = run_pathway_enrichment_for_group(sub, g)

        if res is not None:

            all_pathway_results.append(res)

            out = RESULTS_DIR / f"20-{DATASET_NAME}_Pathway_{g}.csv"
            res.to_csv(out, index=False)

            print(f"[SAVE] {out}")

# -----------------------------
# Combine
# -----------------------------
if all_pathway_results:

    pathway_all_df = pd.concat(all_pathway_results)

    out_all = RESULTS_DIR / f"21-{DATASET_NAME}_Pathway_ALL.csv"
    pathway_all_df.to_csv(out_all, index=False)

    print(f"\n[OUTPUT] {out_all}")

else:
    pathway_all_df = pd.DataFrame()
    print("[WARNING] No pathway results")

print("\n[STEP 20 COMPLETED]\n")

In [ ]:
# n5-23) Visualization (FINAL FIXED)

import matplotlib.pyplot as plt
import numpy as np

print("\n[STEP 21] Visualization...")

if pathway_all_df is None or pathway_all_df.empty:
    print("[WARNING] No data → nothing to plot")

else:

    pathway_all_df["log10_padj"] = -np.log10(pathway_all_df["Adjusted P-value"] + 1e-300)

    for g in pathway_all_df["group"].unique():

        df = pathway_all_df[pathway_all_df["group"] == g].head(10)

        plt.figure()

        plt.scatter(
            df["log10_padj"],
            range(len(df)),
            s=40
        )

        plt.yticks(range(len(df)), df["Term"])

        out = FIG_DIR / f"32-{DATASET_NAME}_pathway_{g}.png"
        plt.savefig(out, bbox_inches="tight")
        plt.close()

        print(f"[FIG] {out}")

print("\n[STEP 21 COMPLETED]\n")

In [ ]:
# END of Notebook 5 - pathway analysis (KEGG / Reactome) 

In [ ]:
# ----------------------

In [ ]:
# Notebook 6 - GSEA

In [ ]:
'''
مرحله	فایل
Setup	n6-24
Helper	n6-25
Run	n6-26
Visualization	n6-27
'''

In [ ]:
# n6-24) GSEA Setup (OFFLINE, ROBUST)

from pathlib import Path
import pandas as pd

print("\n[STEP 22] Initializing GSEA...")

# -----------------------------
# Globals
# -----------------------------
DATASET_NAME = globals().get("DATASET_NAME", "dataset")
RESULTS_DIR = globals().get("RESULTS_DIR", Path("results") / DATASET_NAME)
FIG_DIR = globals().get("FIG_DIR", Path("figures") / DATASET_NAME)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# GSEA gene sets (LOCAL / MSigDB via gseapy)
# -----------------------------
GSEA_GENESETS = [
    "KEGG_2021_Human",
    "Reactome_2022",
    "GO_Biological_Process_2021"
]

# -----------------------------
# Input DEG
# -----------------------------
markers_df = globals().get("markers_df", None)

if markers_df is None or markers_df.empty:
    print("[ERROR] markers_df not found")
    GSEA_READY = False
else:
    GSEA_READY = True
    print(f"[INFO] markers_df shape: {markers_df.shape}")

# detect structure
if GSEA_READY:

    if {"group", "names", "logfoldchanges"}.issubset(markers_df.columns):
        GROUP_COL = "group"
    elif {"celltype", "names", "logfoldchanges"}.issubset(markers_df.columns):
        GROUP_COL = "celltype"
    else:
        print("[ERROR] GSEA requires logfoldchanges column")
        GSEA_READY = False

GENE_COL = "names"
SCORE_COL = "logfoldchanges"

GROUPS = sorted(markers_df[GROUP_COL].dropna().unique()) if GSEA_READY else []

print(f"[INFO] Groups detected: {len(GROUPS)}")

print("[STEP 22 COMPLETED]\n")

In [ ]:
# n6-25) Helper: Run GSEA per group

def run_gsea_for_group(df, group_name):

    import gseapy as gp

    if df is None or df.empty:
        return None

    if SCORE_COL not in df.columns:
        print(f"[GSEA] {group_name} → missing score")
        return None

    # ----------------------------------
    # Fix duplicates (IMPORTANT)
    # ----------------------------------
    df = df.dropna(subset=[GENE_COL, SCORE_COL])
    df = df.groupby(GENE_COL)[SCORE_COL].mean().reset_index()

    # ranking
    rnk = df.sort_values(SCORE_COL, ascending=False)

    if rnk.shape[0] < 50:
        print(f"[GSEA] {group_name} → too few genes")
        return None

    all_results = []

    # ----------------------------------
    # Run GSEA per gene set (CRITICAL FIX)
    # ----------------------------------
    for gs in GSEA_GENESETS:

        print(f"[GSEA] {group_name} → {gs}")

        try:
            pre_res = gp.prerank(
                rnk=rnk,
                gene_sets=gs,   # 🔥 NOT LIST
                threads=4,      # FIX deprecation
                permutation_num=100,
                outdir=None,
                seed=42,
                verbose=False
            )

        except Exception as e:
            print(f"[GSEA ERROR] {group_name} ({gs}): {e}")
            continue

        if pre_res is None or pre_res.res2d is None:
            continue

        tmp = pre_res.res2d.copy()
        tmp["group"] = group_name
        tmp["gene_set"] = gs

        all_results.append(tmp)

    if all_results:
        return pd.concat(all_results)

    return None

In [ ]:
# =============================================================================
# n6-26) Run GSEA for all groups — OFFLINE GMT VERSION (FINAL CLEAN)
# =============================================================================

import gseapy as gp
import pandas as pd
import numpy as np
from pathlib import Path

print("\n[STEP 23] Running GSEA (OFFLINE GMT CLEAN)...")

GSEA_RESULTS = []

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
RESOURCE_DIR = BASE_DIR / "resources"

GMT_FILES = {
    "KEGG": RESOURCE_DIR / "kegg.gmt",
    "GO_BP": RESOURCE_DIR / "go_bp.gmt",
    "REACTOME": RESOURCE_DIR / "reactome.gmt"
}

# check files
valid_gmt = {}
for name, path in GMT_FILES.items():
    if path.exists():
        valid_gmt[name] = path
        print(f"[OK] Loaded {name}: {path}")
    else:
        print(f"[WARNING] Missing {name}: {path}")

if not valid_gmt:
    print("[FATAL] No GMT files found → STOP")
    
# -----------------------------------------------------------------------------
# Loop over groups
# -----------------------------------------------------------------------------
for g in GROUPS:

    print("\n" + "-"*60)
    print(f"[RUN] GSEA → {g}")
    print("-"*60)

    df = markers_df[markers_df[GROUP_COL] == g].copy()

    if df.empty:
        print("[SKIP] Empty group")
        continue

    # -------------------------------------------------------------------------
    # Ranking
    # -------------------------------------------------------------------------
    rank_df = df[["names", "logfoldchanges"]].dropna()

    # remove duplicates
    rank_df = rank_df.groupby("names").mean().reset_index()

    rank_df = rank_df.sort_values("logfoldchanges", ascending=False)

    rnk = rank_df.set_index("names")["logfoldchanges"]

    print(f"[INFO] Genes used: {len(rnk)}")

    if len(rnk) < 50:
        print("[SKIP] Too few genes")
        continue

    # -------------------------------------------------------------------------
    # Run GSEA
    # -------------------------------------------------------------------------
    for lib_name, gmt_file in valid_gmt.items():

        print(f"[GSEA] {g} → {lib_name}")

        try:
            pre_res = gp.prerank(
                rnk=rnk,
                gene_sets=str(gmt_file),
                threads=4,
                min_size=10,
                max_size=500,
                permutation_num=100,
                outdir=None,
                seed=42,
                verbose=False
            )

        except Exception as e:
            print(f"[GSEA ERROR] {g} ({lib_name}): {e}")
            continue

        if pre_res is None or pre_res.res2d is None:
            print("[EMPTY] No result object")
            continue

        res = pre_res.res2d.copy()

        if res.empty:
            print("[EMPTY] No enriched terms")
            continue

        res["group"] = g
        res["library"] = lib_name

        GSEA_RESULTS.append(res)

        print(f"[OK] {g} ({lib_name}) → {res.shape[0]} terms")

# -----------------------------------------------------------------------------
# Combine results
# -----------------------------------------------------------------------------
if GSEA_RESULTS:

    gsea_all_df = pd.concat(GSEA_RESULTS, ignore_index=True)

    out_file = RESULTS_DIR / f"22-{DATASET_NAME}_GSEA_all.csv"
    gsea_all_df.to_csv(out_file, index=False)

    print("\n" + "="*60)
    print("[OUTPUT] GSEA saved:")
    print(out_file)
    print(f"[OUTPUT] Total rows: {gsea_all_df.shape[0]}")
    print("="*60)

else:
    print("\n[WARNING] No GSEA results generated")

print("\n[STEP 23 COMPLETED]")

In [ ]:
# =============================================================================
# n6-27) GSEA Visualization (FINAL CLEAN VERSION)
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("\n[STEP 24] GSEA Visualization...")

if "gsea_all_df" not in globals():
    print("[ERROR] No GSEA data")

elif gsea_all_df.empty:
    print("[WARNING] Empty GSEA table")

else:

    df = gsea_all_df.copy()

    print(f"[DEBUG] Columns: {list(df.columns)}")

    # ---------------------------------------------------------------------
    # Detect TERM column
    # ---------------------------------------------------------------------
    if "Term" in df.columns:
        TERM_COL = "Term"
    elif "Name" in df.columns:
        TERM_COL = "Name"
    else:
        raise ValueError("GSEA term column missing")

    # ---------------------------------------------------------------------
    # Detect FDR column (MUST come BEFORE cleaning)
    # ---------------------------------------------------------------------
    if "fdr" in df.columns:
        FDR_COL = "fdr"
    elif "FDR q-val" in df.columns:
        FDR_COL = "FDR q-val"
    elif "FDR" in df.columns:
        FDR_COL = "FDR"
    else:
        raise ValueError("FDR column missing")

    # ---------------------------------------------------------------------
    # Clean FDR column
    # ---------------------------------------------------------------------
    df[FDR_COL] = pd.to_numeric(df[FDR_COL], errors="coerce")
    df = df.dropna(subset=[FDR_COL])
    df = df[df[FDR_COL] > 0]

    print(f"[DEBUG] After cleaning: {df.shape[0]} rows")

    # ---------------------------------------------------------------------
    # Score
    # ---------------------------------------------------------------------
    df["score"] = -np.log10(df[FDR_COL])

    groups = sorted(df["group"].unique())
    libraries = sorted(df["library"].unique())

    print(f"[INFO] Groups: {groups}")
    print(f"[INFO] Libraries: {libraries}")

    fig_counter = 40
    n_plots = 0

    # ---------------------------------------------------------------------
    # Plot per group × library  ✅ (اصلاح مهم)
    # ---------------------------------------------------------------------
    for g in groups:
        for lib in libraries:

            df_g = (
                df[(df["group"] == g) & (df["library"] == lib)]
                .sort_values("score", ascending=False)
                .head(10)
            )

            if df_g.empty:
                print(f"[WARN] No data for group {g} / {lib}")
                continue

            plt.figure(figsize=(8, 5))

            plt.barh(
                df_g[TERM_COL],
                df_g["score"]
            )

            # -----------------------------------------------------------------
            # 🔥 TITLE FIX (مهم‌ترین تغییر)
            # -----------------------------------------------------------------
            plt.title(f"GSEA - {lib} (Group {g})")

            plt.xlabel("-log10(FDR)")
            plt.gca().invert_yaxis()  # بهتر دیده شدن top term

            # -----------------------------------------------------------------
            # 🔥 FILE NAME FIX
            # -----------------------------------------------------------------
            safe_lib = lib.replace(" ", "_")

            path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GSEA_{safe_lib}_G{g}.png"

            plt.savefig(path, dpi=300, bbox_inches="tight")
            plt.close()

            print(f"[FIG] Saved → {path}")

            fig_counter += 1
            n_plots += 1

    # ---------------------------------------------------------------------
    # Summary
    # ---------------------------------------------------------------------
    if n_plots == 0:
        print("[WARNING] No plots generated")
    else:
        print(f"[INFO] Total plots generated: {n_plots}")

print("\n[STEP 24 COMPLETED]")

In [ ]:
## END of Notebook 6 - GSEA

In [ ]:
## ------------------

In [ ]:
# Notebook 7 - Biological Interpretation

In [ ]:
# تحلیل بیولوژیک کلاستر هشتم چون در آن ژنهایی که ترشح ایمونوکلوبولین دارند زیاد فعال شده بودند و میتوان یک داستان و فرضیه برای آن ساخت.

In [ ]:
# =============================================================================
# n7-1) Select cluster for deep GSEA analysis (Cluster 0)
# =============================================================================

CLUSTER_ID = "0"   # ← انتخاب بهینه (سیگنال قوی‌تر)

df_cluster = gsea_all_df[gsea_all_df["group"] == CLUSTER_ID].copy()

print(f"[INFO] Rows in cluster {CLUSTER_ID}: {df_cluster.shape[0]}")
df_cluster.head()

In [ ]:
# =============================================================================
# n7-2) Filter significant pathways (FDR < 0.25)
# =============================================================================

FDR_COL = "FDR q-val"

# تبدیل به عدد (حل مشکل log10)
df_cluster[FDR_COL] = pd.to_numeric(df_cluster[FDR_COL], errors="coerce")

# حذف مقادیر نامعتبر
df_cluster = df_cluster.dropna(subset=[FDR_COL])

# فیلتر استاندارد GSEA
df_sig = df_cluster[df_cluster[FDR_COL] < 0.25].copy()

print(f"[INFO] Significant pathways: {df_sig.shape[0]}")

In [ ]:
# =============================================================================
# n7-3) Show top pathways per library
# =============================================================================

for lib in df_sig["library"].unique():

    print("\n" + "="*50)
    print(f"[TOP PATHWAYS] {lib}")
    print("="*50)

    display(
        df_sig[df_sig["library"] == lib]
        .sort_values("NES", ascending=False)
        [["Term", "NES", "FDR q-val"]]
        .head(10)
    )

In [ ]:
# =============================================================================
# n7-4) GSEA Visualization (Cluster 0 - CLEAN + LABELED)
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np

plt.ioff()  # جلوگیری از نمایش در نوتبوک

fig_counter = 70
n_plots = 0

# محاسبه score ایمن
df_sig["score"] = -np.log10(df_sig[FDR_COL] + 1e-300)

for lib in df_sig["library"].unique():

    df_plot = (
        df_sig[df_sig["library"] == lib]
        .sort_values("score", ascending=False)
        .head(10)
    )

    if df_plot.empty:
        continue

    plt.figure(figsize=(7, 5))

    plt.barh(
        df_plot["Term"],
        df_plot["score"]
    )

    plt.title(f"GSEA - Cluster {CLUSTER_ID} ({lib})")
    plt.xlabel("-log10(FDR)")

    path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GSEA_cluster{CLUSTER_ID}_{lib}.png"

    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"GSEA plot saved to: {path}")

    fig_counter += 1
    n_plots += 1

print(f"[INFO] Total plots generated: {n_plots}")

In [ ]:
# =============================================================================
# n7-5) Extract leading edge genes
# =============================================================================

top_terms = (
    df_sig
    .sort_values("NES", ascending=False)
    .head(5)
)

genes = []

for g in top_terms["Lead_genes"]:
    genes.extend(str(g).split(";"))

top_genes = list(set(genes))

print("[TOP GENES]")
print(top_genes[:30])

In [ ]:
# =============================================================================
# BIOLOGICAL INTERPRETATION (Cluster 0)
# =============================================================================
#
# Cluster 0 shows enrichment in pathways related to:
#   - Immune response / inflammation (if present)
#   - Active transcription / translation (check NES direction)
#   - Possibly monocyte/macrophage activation signatures
#
# Compared to Cluster 8 (translation downregulation),
# Cluster 0 likely represents a more metabolically active
# and biologically functional cell population.
#
# This makes it a better candidate for biological interpretation
# and downstream analysis in a publication or GitHub portfolio.
#
# Key interpretation strategy:
#   - Positive NES → activated pathways
#   - Negative NES → suppressed pathways
#
# Recommended next step:
#   - Map pathways to known cell types (monocytes, macrophages, etc.)
#   - Integrate with marker genes
#
# =============================================================================

In [ ]:
## END of Notebook 7 - Biological Interpretation

In [ ]:
## -----------------------------------

In [ ]:
## Notebook 8 - Figures

In [ ]:
# =============================================================================
# n8-1) NES Heatmap (Publication-Level FIXED)
# =============================================================================

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

plt.ioff()

# ---------------------------------------------------------------------
# Clean data
# ---------------------------------------------------------------------
df = gsea_all_df.copy()

# تبدیل NES به عدد
df["NES"] = pd.to_numeric(df["NES"], errors="coerce")

# حذف مقادیر خراب
df = df.dropna(subset=["NES", "Term", "group"])

# ---------------------------------------------------------------------
# انتخاب مسیرهای مهم (TOP terms)
# ---------------------------------------------------------------------
top_terms = (
    df.sort_values("FDR q-val")
    .groupby("library")
    .head(8)   # هر library چند pathway
)

# ---------------------------------------------------------------------
# Pivot برای heatmap
# ---------------------------------------------------------------------
heatmap_df = top_terms.pivot_table(
    index="Term",
    columns="group",
    values="NES",
    aggfunc="mean"
)

# حذف سطرهای کاملاً خالی
heatmap_df = heatmap_df.dropna(how="all")

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
plt.figure(figsize=(12, 9))

sns.heatmap(
    heatmap_df,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    linecolor="gray",
    cbar_kws={"label": "NES"}
)

plt.title("Functional Landscape of Cell Clusters (GSEA NES)", fontsize=14)
plt.xlabel("Cluster")
plt.ylabel("Pathways")

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------
path = FIG_DIR / f"80-{DATASET_NAME}_NES_heatmap.png"

plt.savefig(path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Heatmap saved to: {path}")

In [ ]:
# =============================================================================
# n8-2) Top Pathways Barplot (Cluster 0 - Publication)
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np

plt.ioff()

CLUSTER_ID = "0"

df = gsea_all_df.copy()

# Clean
df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")
df = df.dropna(subset=["FDR q-val"])

# انتخاب cluster
df = df[df["group"] == CLUSTER_ID]

# انتخاب بهترین مسیرها
df_top = (
    df.sort_values("FDR q-val")
    .head(15)
)

# score
df_top["score"] = -np.log10(df_top["FDR q-val"] + 1e-300)

plt.figure(figsize=(8, 6))

plt.barh(
    df_top["Term"],
    df_top["score"]
)

plt.gca().invert_yaxis()

plt.xlabel("-log10(FDR)")
plt.title(f"Top Enriched Pathways (Cluster {CLUSTER_ID})")

path = FIG_DIR / f"81-{DATASET_NAME}_TopPathways_cluster{CLUSTER_ID}.png"

plt.savefig(path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Barplot saved to: {path}")

In [ ]:
# =============================================================================
# BIOLOGICAL INSIGHT (Figure Interpretation)
# =============================================================================
#
# The heatmap reveals distinct functional programs across clusters,
# highlighting both activated and suppressed pathways.
#
# Cluster 0 shows strong enrichment in biologically active pathways,
# suggesting a metabolically active and functionally relevant cell population.
#
# In contrast, clusters such as Cluster 8 display suppression of
# translation-related pathways, indicating a more quiescent or stressed state.
#
# This functional heterogeneity supports the presence of multiple
# monocyte/macrophage states within the dataset.
#
# =============================================================================

In [ ]:
# =============================================================================
# # n8-3) FIGURE 82 — GSEA NES Heatmap (CLEAN + PUBLICATION READY)
# =============================================================================

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

print("\n[FIGURE 80] NES Heatmap...")

df = gsea_all_df.copy()

# ---------------------------------------------------------------------
# Clean numeric columns
# ---------------------------------------------------------------------
df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
df = df.dropna(subset=["NES"])

# ---------------------------------------------------------------------
# Select top pathways (per group)
# ---------------------------------------------------------------------
top_terms = (
    df.sort_values("FDR q-val")
    .groupby("group")
    .head(5)
)

# ---------------------------------------------------------------------
# Pivot
# ---------------------------------------------------------------------
heatmap_df = top_terms.pivot_table(
    index="Term",
    columns="group",
    values="NES",
    aggfunc="mean"
)

# ensure numeric
heatmap_df = heatmap_df.astype(float)

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
plt.figure(figsize=(10, 8))

sns.heatmap(
    heatmap_df,
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("GSEA NES Heatmap Across Clusters")
plt.xlabel("Cluster")
plt.ylabel("Pathways")

path = FIG_DIR / f"82-{DATASET_NAME}_NES_heatmap.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] Saved → {path}")

In [ ]:
# =============================================================================
#  n8-4) - FIGURE 83 — Bubble Plot (GSEA)
# =============================================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("\n[FIGURE 83] Bubble Plot...")

df = gsea_all_df.copy()

# clean
df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")

df = df.dropna(subset=["NES", "FDR q-val"])
df = df[df["FDR q-val"] > 0]

df["score"] = -np.log10(df["FDR q-val"])

# select top
df_top = (
    df.sort_values("FDR q-val")
    .groupby(["group", "library"])
    .head(5)
)

plt.figure(figsize=(12, 8))

plt.scatter(
    x=df_top["group"],
    y=df_top["Term"],
    s=df_top["score"] * 50,
    c=df_top["NES"],
    cmap="coolwarm"
)

plt.colorbar(label="NES")
plt.xlabel("Cluster")
plt.ylabel("Pathway")
plt.title("GSEA Bubble Plot")

path = FIG_DIR / f"83-{DATASET_NAME}_bubble_plot.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] Saved → {path}")

In [ ]:
# =============================================================================
#  n8-5) - FIGURE 84 — Barplot (Top Pathways Cluster 0)
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np

print("\n[FIGURE 84] Barplot...")

CLUSTER_ID = "0"

df = gsea_all_df.copy()

df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")

df = df.dropna(subset=["NES", "FDR q-val"])
df = df[df["FDR q-val"] > 0]

df["score"] = -np.log10(df["FDR q-val"])

df_c = df[df["group"] == CLUSTER_ID]

top = df_c.sort_values("score", ascending=False).head(10)

plt.figure(figsize=(8, 6))

plt.barh(
    top["Term"],
    top["score"]
)

plt.gca().invert_yaxis()

plt.xlabel("-log10(FDR)")
plt.title(f"Top Enriched Pathways (Cluster {CLUSTER_ID})")

path = FIG_DIR / f"84-{DATASET_NAME}_barplot_cluster{CLUSTER_ID}.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] Saved → {path}")

In [ ]:
# =============================================================================
#  n8-6) - FIGURE 83 — FINAL MULTI-PANEL GSEA FIGURE (PUBLICATION READY)
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("\n[FIGURE 85] Final multi-panel figure...")

df = gsea_all_df.copy()

# ---------------------------------------------------------------------
# CLEAN DATA
# ---------------------------------------------------------------------
df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")

df = df.dropna(subset=["NES", "FDR q-val"])
df = df[df["FDR q-val"] > 0]

df["score"] = -np.log10(df["FDR q-val"])

# ---------------------------------------------------------------------
# CREATE FIGURE LAYOUT
# ---------------------------------------------------------------------
fig = plt.figure(figsize=(18, 12))

# =========================================================
# PANEL A — HEATMAP
# =========================================================
ax1 = plt.subplot2grid((2, 2), (0, 0))

top_terms = (
    df.sort_values("FDR q-val")
    .groupby("group")
    .head(5)
)

heatmap_df = top_terms.pivot_table(
    index="Term",
    columns="group",
    values="NES",
    aggfunc="mean"
).astype(float)

sns.heatmap(
    heatmap_df,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    ax=ax1
)

ax1.set_title("A) NES Heatmap Across Clusters", fontsize=12)
ax1.set_xlabel("Cluster")
ax1.set_ylabel("Pathway")

# =========================================================
# PANEL B — BUBBLE PLOT
# =========================================================
ax2 = plt.subplot2grid((2, 2), (0, 1))

df_top = (
    df.sort_values("FDR q-val")
    .groupby(["group", "library"])
    .head(3)
)

scatter = ax2.scatter(
    x=df_top["group"],
    y=df_top["Term"],
    s=df_top["score"] * 40,
    c=df_top["NES"],
    cmap="coolwarm"
)

fig.colorbar(scatter, ax=ax2, label="NES")

ax2.set_title("B) GSEA Bubble Plot", fontsize=12)
ax2.set_xlabel("Cluster")
ax2.set_ylabel("Pathway")

# =========================================================
# PANEL C — BARPLOT (Cluster 0)
# =========================================================
ax3 = plt.subplot2grid((2, 2), (1, 0), colspan=2)

CLUSTER_ID = "0"

df_c = df[df["group"] == CLUSTER_ID]

top = df_c.sort_values("score", ascending=False).head(10)

ax3.barh(
    top["Term"],
    top["score"]
)

ax3.invert_yaxis()

ax3.set_title(f"C) Top Pathways (Cluster {CLUSTER_ID})", fontsize=12)
ax3.set_xlabel("-log10(FDR)")

# =========================================================
# FINAL TOUCH
# =========================================================
plt.tight_layout()

path = FIG_DIR / f"85-{DATASET_NAME}_FINAL_multi_panel.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] Saved → {path}")

In [ ]:
## END of Notebook 8 - Figures